# Amazon ML Challenge 2026: Business Entity Resolution
### Multilingual Normalization + Static Lexicon, Multi-View Blocking, Two-Stage GBDT (XGBoost on GPU)

**Pipeline**:
- **Data**: read directly from the attached Kaggle dataset (`/kaggle/input/...`), no downloads.
- **Normalisation**: all major Indic scripts + Urdu romanised, French ligatures/apostrophes, legal forms, landmarks,
  PIN/ZIP parsing, plus a static per-country lexicon (Indic-script English loanwords, abbreviations, state codes,
  typos, OCR digit noise) embedded in cell 7. Built offline once; no API is called here.
- **Training data**: every record of whole regions (density preserved), split 60/20/20 by Source 1.
- **Blocking**: 3-view country-partitioned char TF-IDF; vectorisers fitted once per country.
- **Matching**: stage-1 GBDT on pair features -> stage-2 GBDT with group context (out-of-fold); XGBoost on GPU, LightGBM on CPU-only machines.
- **Selection**: two thresholds (tau1, tau2) tuned on Macro F0.5, globally one-to-one.
- **Inference**: Source 1 chunks, stage-2 inputs cached on disk (float16), all 1.73M Source 1 rows written.

Set `DEV_MODE = True` in the config cell for a quick sanity run.

In [ ]:
# [SETUP 1] Environment Verification (Zero Network Required, Pre-installed Packages Respected)
import sys, os, importlib.util

missing = []
for mod, pkg in [("rapidfuzz", "rapidfuzz>=3.6.0"), ("indic_transliteration", "indic-transliteration>=2.3.0")]:
    if not importlib.util.find_spec(mod):
        missing.append(pkg)
import shutil as _sh
if _sh.which("nvidia-smi") and not importlib.util.find_spec("cupy"):
    missing.append("cupy-cuda12x")

if missing:
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing, "-q", "--timeout", "5"])
    except Exception as e:
        print(f"Note: Running in offline mode without extra optional packages ({missing}). Native fallbacks will be used.")
print("Python packages ready.")
try:
    import cupy as _cp
    print(f"GPU: {_cp.cuda.runtime.getDeviceCount()} CUDA device(s) -> TF-IDF blocking (and XGBoost) run on GPU")
except Exception as _e:
    print(f"GPU not available ({type(_e).__name__}) -> blocking runs on CPU")
print(f"CPU cores: {os.cpu_count()} -> text cleaning and pair features run in parallel")

In [ ]:
# [SETUP 2] Standard Imports
import os, sys, csv, time, json, zipfile, re, unicodedata, subprocess, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer

from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist

try:
    from indic_transliteration import sanscript as _S
    HAS_TRANSLIT = True
except Exception:
    _S = None
    HAS_TRANSLIT = False

print("Libraries imported successfully.")

In [ ]:
# [CONFIG] Global Configuration & Direct Dataset Path Discovery
DEV_MODE = False  # Set to False for complete 1.73M test submission run
os.environ.setdefault("ER_USE_GPU", "1")      # GPU blocking (CPU fallback is automatic)
os.environ.setdefault("ER_BACKEND", "auto")   # auto: XGBoost on GPU if present, else LightGBM

IS_KAGGLE = Path("/kaggle/input").exists()
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
OUTPUT_DIR = WORKING_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_tsv(filename, dev=False):
    roots = []
    if IS_KAGGLE:
        roots.extend([
            Path("/kaggle/input/zamzon-2026/student_resource/dataset"),
            Path("/kaggle/input/zamzon-2026/student_resource"),
            Path("/kaggle/input/zamzon-2026"),
            Path("/kaggle/input"),
        ])
    roots.extend([
        Path.cwd() / "dataset",
        Path.cwd(),
        Path.cwd().parent / "dataset",
        Path.cwd().parent,
    ])
    
    if dev:
        for r in roots:
            if r.exists():
                for p in r.rglob(filename):
                    if "dev" in p.parts and ".venv" not in p.parts:
                        return p
    for r in roots:
        if r.exists():
            for p in r.rglob(filename):
                if "dev" not in p.parts and ".venv" not in p.parts:
                    return p
    return None

train_s1_path = find_tsv("train_source1.tsv", dev=DEV_MODE)
train_s2_path = find_tsv("train_source2.tsv", dev=DEV_MODE)
train_s3_path = find_tsv("train_source3.tsv", dev=DEV_MODE)
train_gt_path = find_tsv("train_ground_truth.tsv", dev=DEV_MODE)

test_s1_path  = find_tsv("test_source1.tsv", dev=DEV_MODE)
test_s2_path  = find_tsv("test_source2.tsv", dev=DEV_MODE)
test_s3_path  = find_tsv("test_source3.tsv", dev=DEV_MODE)
val_script    = find_tsv("validate_submission.py")

print("Dataset Paths Discovered (Loaded directly from input):")
print(f"  Train S1: {train_s1_path}")
print(f"  Train S2: {train_s2_path}")
print(f"  Train S3: {train_s3_path}")
print(f"  Train GT: {train_gt_path}")
print(f"  Test S1:  {test_s1_path}")
print(f"  Test S2:  {test_s2_path}")
print(f"  Test S3:  {test_s3_path}")
print(f"  Validator:{val_script}")

# Safety assertions
for p, name in [(train_s1_path, "train_s1"), (test_s1_path, "test_s1")]:
    assert p is not None and p.exists(), f"CRITICAL: {name} path not found in mounted directories!"
print("Path verification complete.")

In [ ]:
import os
import csv
from pathlib import Path
import pandas as pd

def read_tsv(path):
    """Read a challenge TSV safely (one record per line).
    
    Prevents unclosed quote character swallowing and 'NA' string nullification.
    Asserts exact line count and entity_id uniqueness.
    """
    path = Path(path)
    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        quoting=csv.QUOTE_NONE,
        keep_default_na=False,
        na_values=[],
        encoding="utf-8"
    )
    with open(path, "r", encoding="utf-8") as f:
        next(f, None)
        n_lines = sum(1 for line in f if line.strip())
    assert len(df) == n_lines, f"{path.name}: parsed {len(df)} rows, file has {n_lines} data lines"
    assert df.iloc[:, 0].is_unique, f"{path.name}: duplicate IDs in first column"
    return df

def find_file(filename, dev_mode=False):
    """Locate dataset TSV files across Kaggle, student_resource, and repo folders."""
    roots = []
    if Path("/kaggle/input").exists():
        roots.extend([
            Path("/kaggle/input/zamzon-2026/student_resource/dataset"),
            Path("/kaggle/input/zamzon-2026/student_resource"),
            Path("/kaggle/input/zamzon-2026/dataset"),
            Path("/kaggle/input/zamzon-2026"),
            Path("/tmp/dataset"),
            Path("/kaggle/input"),
        ])
    
    roots.extend([
        Path.cwd() / "dataset",
        Path.cwd(),
        Path.cwd().parent / "dataset",
        Path.cwd().parent,
    ])
    
    if dev_mode:
        for r in roots:
            if r.exists():
                for p in r.rglob(filename):
                    if "dev" in p.parts and ".venv" not in p.parts:
                        return p
                        
    for r in roots:
        if r.exists():
            for p in r.rglob(filename):
                if "dev" not in p.parts and ".venv" not in p.parts:
                    return p
    return None

In [ ]:
import re
import unicodedata
import pandas as pd

try:
    from indic_transliteration.sanscript import transliterate, DEVANAGARI, HK
    HAS_TRANSLIT = True
except Exception:
    HAS_TRANSLIT = False

_STOP = set("""
pvt private ltd limited llc llp inc incorporated corp corporation co company plc
the and of
sarl sas sasu sa eurl snc sci scp selarl ets etablissements societe cie et
de du des la le les l d au aux
""".split())

def canon_country(c):
    if not c or pd.isna(c):
        return ""
    s = str(c).strip().lower()
    s = re.sub(r'[^a-z0-9]', '', s)
    if s in {"us", "usa", "unitedstates", "unitedstatesofamerica"}:
        return "US"
    if s in {"in", "ind", "india", "bharat"}:
        return "INDIA"
    if s in {"fr", "fra", "france", "french", "republiquefrancaise"}:
        return "FRANCE"
    return s.upper()

def transliterate_text(text):
    if not HAS_TRANSLIT or not isinstance(text, str):
        return text
    if any('\u0900' <= ch <= '\u097f' for ch in text):
        try:
            return transliterate(text, DEVANAGARI, HK)
        except Exception:
            return text
    return text

def normalize_text(text):
    if not text or pd.isna(text):
        return ""
    s = transliterate_text(str(text))
    s = unicodedata.normalize('NFKD', s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    # Replace & with and
    s = re.sub(r'&', ' and ', s)
    # Remove unwanted punctuation
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def core_name(norm_name):
    toks = [t for t in str(norm_name).split() if t not in _STOP]
    return " ".join(toks) if toks else str(norm_name)

def _initials(s):
    toks = str(s).split()
    return "".join(t[0] for t in toks) if len(toks) >= 2 else ""

In [ ]:
# er_multilingual.py - multilingual / country-specific text handling for the ER pipeline.
"""
er_multilingual.py - multilingual / country-specific text handling for the ER pipeline.

Replaces the notebook's normalize_text, which (verified):
  * turns Tamil, Telugu, Bengali, Gujarati, Gurmukhi, Malayalam, Odia and Urdu text into ""
    -> two unrelated wiped names then score 1.0 on Jaro-Winkler, Levenshtein and Jaccard
  * romanises Devanagari/Kannada with Harvard-Kyoto: "शर्मा ट्रेडर्स" -> "zarma tredarsa"
  * drops the French ligature: "Cœur" -> "cur"
  * treats curly and straight apostrophes differently: "L’Atelier" -> "latelier", "L'Atelier" -> "l atelier"
  * maps every "st" to "street", so "Pharmacie St-Denis" -> "pharmacie street denis"
  * splits Indian PINs written "560 001" into two tokens

Design rule: every mapping here is symmetric and language-agnostic (applied to all rows
regardless of the country label), so it works unchanged for an unseen country.

Requires: indic-transliteration (MIT), rapidfuzz.
"""
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

try:
    from indic_transliteration import sanscript as _S
except Exception:  # pragma: no cover
    _S = None

# --------------------------------------------------------------------------------------
# Script detection and audit
# --------------------------------------------------------------------------------------
_SCRIPTS = [  # (name, lo, hi, sanscript scheme, schwa-deleting language?)
    ("devanagari", 0x0900, 0x097F, "DEVANAGARI", True),
    ("bengali", 0x0980, 0x09FF, "BENGALI", True),
    ("gurmukhi", 0x0A00, 0x0A7F, "GURMUKHI", True),
    ("gujarati", 0x0A80, 0x0AFF, "GUJARATI", True),
    ("oriya", 0x0B00, 0x0B7F, "ORIYA", True),
    ("tamil", 0x0B80, 0x0BFF, "TAMIL", False),
    ("telugu", 0x0C00, 0x0C7F, "TELUGU", False),
    ("kannada", 0x0C80, 0x0CFF, "KANNADA", False),
    ("malayalam", 0x0D00, 0x0D7F, "MALAYALAM", False),
    ("arabic", 0x0600, 0x06FF, None, False),
]


def script_of(ch):
    o = ord(ch)
    if o < 0x80:
        return "ascii"
    for name, lo, hi, _, _ in _SCRIPTS:
        if lo <= o <= hi:
            return name
    if 0x00C0 <= o <= 0x024F:
        return "latin_accented"
    return "other"


def script_audit(frames, cols=("business_name", "business_address")):
    """frames: {"train_s1": df, ...} with a country column. Share of records containing each
    script, per source x country x column. Run on train AND test before building anything."""
    rows = []
    for src, df in frames.items():
        for col in cols:
            present = df[col].fillna("").map(lambda s: frozenset(script_of(c) for c in s if c.isalpha()))
            for ctry, grp in present.groupby(df["country"].fillna("")):
                cnt = Counter(s for fs in grp for s in fs)
                for s, n in cnt.items():
                    if s != "ascii":
                        rows.append((src, ctry, col, s, n / len(grp)))
    return (pd.DataFrame(rows, columns=["source", "country", "column", "script", "share"])
            .sort_values("share", ascending=False).reset_index(drop=True))


def vocab_shift(texts_a, texts_b, top=25, min_count=20):
    """Tokens over-represented in A vs B (log ratio). Run per country between sources, e.g.
    France S1 names vs France S3 names: English business words on one side and French on the
    other means a translation-type variation exists and needs handling."""
    ca = Counter(t for s in texts_a for t in str(s).split())
    cb = Counter(t for s in texts_b for t in str(s).split())
    na, nb = sum(ca.values()) or 1, sum(cb.values()) or 1
    toks = [t for t in set(ca) | set(cb) if ca[t] + cb[t] >= min_count]
    lr = pd.Series({t: np.log((ca[t] + 1) / na) - np.log((cb[t] + 1) / nb) for t in toks})
    return lr.nlargest(top).rename("more_in_A"), lr.nsmallest(top).rename("more_in_B")


# --------------------------------------------------------------------------------------
# Romanisation
# --------------------------------------------------------------------------------------
_CHARMAP = str.maketrans({
    "œ": "oe", "Œ": "OE", "æ": "ae", "Æ": "AE", "ß": "ss", "ø": "o", "Ø": "O", "ł": "l", "Ł": "L",
    "đ": "d", "Đ": "D", "ð": "d", "þ": "th", "ı": "i",
    "’": "'", "‘": "'", "‛": "'", "`": "'", "´": "'", "ʼ": "'",
    "“": '"', "”": '"', "„": '"', "«": '"', "»": '"',
    "‐": "-", "‑": "-", "‒": "-", "–": "-", "—": "-", "―": "-", "№": " no ", "°": " ", "º": "", "ª": "",
})
# Urdu/Arabic: consonant skeleton only (the script omits short vowels, so keys below still work)
_ARABIC = str.maketrans({
    "ا": "a", "آ": "a", "ب": "b", "پ": "p", "ت": "t", "ٹ": "t", "ث": "s", "ج": "j", "چ": "ch", "ح": "h",
    "خ": "kh", "د": "d", "ڈ": "d", "ذ": "z", "r": "r", "ڑ": "r", "ز": "z", "ژ": "zh", "س": "s", "ش": "sh",
    "ص": "s", "ض": "z", "ط": "t", "ظ": "z", "ع": "", "غ": "gh", "ف": "f", "ق": "q", "ک": "k", "ك": "k",
    "گ": "g", "ل": "l", "م": "m", "ن": "n", "ں": "n", "و": "o", "ہ": "h", "ه": "h", "ھ": "h", "ء": "",
    "ی": "i", "ي": "i", "ے": "e", "ئ": "i", "ؤ": "o", "ة": "a",
})
_UNIT_RX = r"(?:[kgcCjJTDtdpbsS]h|~n|[bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ])"  # ITRANS consonant units


def _itrans_cleanup(tok, schwa):
    t = tok.replace("RRi", "ri").replace("R^i", "ri").replace("j~n", "gy").replace("~n", "n")
    t = re.sub(r"M(?=[pbmPB])", "m", t).replace("M", "n").replace(".N", "n").replace(".n", "n")
    t = re.sub(r"[\^~.]", "", t)
    if schwa:  # word-final inherent 'a' is silent in Hindi/Gujarati/Bengali/Punjabi: rAma -> ram,
        # but kept after a consonant cluster (kRShNa -> krishna, gupta, mishra) unless it ends in s (TreDarsa -> treDars)
        m = re.search(rf"((?:{_UNIT_RX})+)a$", t)
        if m and len(re.findall(r"[aeiouAEIOU]", t)) > 1:
            units = re.findall(_UNIT_RX, m.group(1))
            if len(units) == 1 or units[-1] == "s":
                t = t[:-1]
    return t.replace("x", "ksh").lower()


def _unicode_name_fallback(ch):
    try:
        nm = unicodedata.name(ch)
    except ValueError:
        return ""
    if "LETTER" in nm or "VOWEL SIGN" in nm:
        w = nm.split()[-1].lower()
        w = w[:-1] if len(w) > 1 and w.endswith("a") and "VOWEL" not in nm else w
        return re.sub(r"(.)\1+", r"\1", w)
    return ""


def to_latin(text):
    """Any script -> lowercase ASCII-ish Latin, never silently dropping letters."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = unicodedata.normalize("NFKC", text).translate(_CHARMAP)
    if any(ord(c) > 0x5FF for c in t):
        out = []
        for tok in t.split():
            scripts = {script_of(c) for c in tok if c.isalpha()}
            for name, _, _, scheme, schwa in _SCRIPTS:
                if name in scripts and scheme and _S is not None:
                    tok = _itrans_cleanup(_S.transliterate(tok, getattr(_S, scheme), _S.ITRANS), schwa)
            if "arabic" in scripts:
                tok = tok.translate(_ARABIC)
            out.append(tok)
        t = " ".join(out)
    t = "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))
    if any(ord(c) > 127 for c in t):  # leftover letters (e.g. Tamil final consonants)
        t = "".join(c if ord(c) < 128 else _unicode_name_fallback(c) for c in t)
    return t.lower()


# --------------------------------------------------------------------------------------
# Canonical tokens (symmetric: both spellings map to one token; collisions are harmless)
# --------------------------------------------------------------------------------------
_PHRASES = [  # multi-word first
    (r"\bprivate limited\b", "pvt ltd"), (r"\bdoing business as\b|\bd b a\b", "dba"),
    (r"\btrading as\b|\bt a\b", "dba"), (r"\ba k a\b|\balso known as\b", "aka"),
    (r"\bzone industrielle\b", "zi"), (r"\bzone d activites?\b|\bzone artisanale\b", "za"),
    (r"\bcentre commercial\b|\bshopping (?:centre|center|mall)\b", "cc"),
    (r"\blieu dit\b", "ld"), (r"\bin front of\b|\ben face de\b|\bface a\b", "opp"),
    (r"\bnext to\b|\ba cote de\b|\bclose to\b|\bpres de\b", "near"),
    (r"\bunited states(?: of america)?\b", "usa"),
    (r"\b(?:h|house|door|d|plot|shop|flat|bldg) no\b", "no"),
]
_CANON = {}
for canon, variants in {
    # legal / company words (EN, IN, FR)
    "co": "co company compagnie cie", "corp": "corp corporation", "inc": "inc incorporated",
    "ltd": "ltd limited", "pvt": "pvt private", "llc": "llc", "and": "and et &",
    "bros": "bros brothers freres", "intl": "intl international", "mfg": "mfg manufacturing",
    "svc": "svc svcs services service", "ent": "ent enterprises enterprise entreprise",
    "ind": "ind inds industries industry", "tech": "tech technologies technology",
    "assoc": "assoc associates associes", "mgmt": "mgmt management", "engg": "engg engineering",
    "ets": "ets etablissements etablissement", "ste": "ste suite sainte societe",
    "st": "st saint street str", "mkt": "mkt market bazaar bazar",
    # street types
    "rd": "rd road raod", "ave": "ave av avenue", "blvd": "blvd bd boulevard boul",
    "dr": "dr drive", "ln": "ln lane gali gully", "ct": "ct court", "pl": "pl place",
    "hwy": "hwy highway", "pkwy": "pkwy parkway", "sq": "sq square", "rte": "rte route",
    "imp": "imp impasse", "chem": "chem chemin", "fbg": "fbg fg faubourg", "res": "res residence",
    "bldg": "bldg building batiment bat", "apt": "apt apts apartment apartments appt",
    "fl": "fl floor etage", "sec": "sec sector", "ph": "ph phase", "extn": "extn ext extension",
    "nagar": "nagar ngr nager", "layout": "layout lyt", "colony": "colony col clny",
    "dist": "dist district dt", "vill": "vill village vil", "taluk": "taluk taluka tq",
    "opp": "opp opposite", "near": "near nr", "behind": "behind bhd",
    # directions
    "n": "n north", "s": "s south", "e": "e east", "w": "w west",
}.items():
    for v in variants.split():
        _CANON[v] = canon
_NUMWORDS = {w: str(i) for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen "
    "sixteen seventeen eighteen nineteen twenty".split())}
_NUMWORDS.update({w: str(i) for i, w in enumerate(
    "zeroth first second third fourth fifth sixth seventh eighth ninth tenth eleventh twelfth".split())})
_NUMWORDS.update({"premier": "1", "premiere": "1", "deuxieme": "2", "troisieme": "3", "quatrieme": "4",
                  "cinquieme": "5", "sixieme": "6", "septieme": "7", "huitieme": "8", "neuvieme": "9",
                  "dixieme": "10", "bis": "b", "ter": "t"})
_DROP = {"no", "num", "number", "cedex", "bp", "usa", "india", "bharat", "france"}
_PREFIX = re.compile(r"^\s*(?:m/s\.?|messrs\.?|m\.s\.)\s*", re.I)


def normalize_ml(text, kind="name"):
    """kind: 'name' or 'addr'. Returns canonical lowercase tokens joined by spaces."""
    t = to_latin(text)
    if not t:
        return ""
    if kind == "name":
        t = _PREFIX.sub("", t)
    t = re.sub(r"'s\b", "", t)                          # joe's -> joe
    t = re.sub(r"(\d)\s*[-/]\s*(?=\d)", r"\1-", t)      # 12 - 3 / 456 -> 12-3-456
    t = re.sub(r"(\d{5})-\d{4}\b", r"\1", t)            # ZIP+4 -> ZIP
    t = re.sub(r"[^a-z0-9\-&]+", " ", t.replace("&", " & "))
    t = re.sub(r"(?<![0-9])-|-(?![0-9])", " ", t)       # keep hyphens only between digits
    if kind == "addr":
        t = re.sub(r"\b([1-9]\d{2})\s+(\d{3})\b", r"\1\2", t)   # PIN "560 001" -> "560001"
        t = re.sub(r"\bcedex\s*\d*\b", " ", t)
    t = re.sub(r"\b(\d+)(?:st|nd|rd|th|er|ere|e|eme|ieme)\b", r"\1", t)  # ordinals: 5th / 2eme -> 5 / 2
    t = re.sub(r"\b(\d+)\s*(bis|ter)\b", lambda m: m.group(1) + m.group(2)[0], t)  # 12 bis -> 12b
    t = re.sub(r"\b([a-z])\s+(?=[a-z]\b)", r"\1", t)     # dotted initials: "m g rd" -> "mg rd", "a b c" -> "abc"
    for pat, rep in _PHRASES:
        t = re.sub(pat, rep, t)
    toks = []
    for w in t.split():
        w = _NUMWORDS.get(w, w)
        w = _CANON.get(w, w)
        if kind == "addr" and w in _DROP:
            continue
        toks.append(w)
    return " ".join(toks)


# --------------------------------------------------------------------------------------
# Phonetic keys (feature + optional extra blocking key)
# --------------------------------------------------------------------------------------
_IND_RULES = [("ksh", "x"), ("ks", "x"), ("chh", "c"), ("ch", "c"), ("sh", "s"), ("th", "t"),
              ("dh", "d"), ("bh", "b"), ("ph", "f"), ("kh", "k"), ("gh", "g"), ("jh", "j"),
              ("ow", "o"), ("au", "o"), ("ou", "o"), ("ee", "i"), ("oo", "u"), ("aa", "a"),
              ("w", "v"), ("z", "j"), ("q", "k"), ("ck", "k"), ("y", "i"),
              ("g", "k"), ("d", "t"), ("b", "p")]          # Tamil-style voicing merges
_FR_RULES = [("eaux", "o"), ("eau", "o"), ("aux", "o"), ("ault", "o"), ("au", "o"), ("ph", "f"),
             ("qu", "k"), ("ch", "s"), ("gn", "n"), ("th", "t"), ("h", ""), ("y", "i"),
             ("ai", "e"), ("ei", "e"), ("ez", "e"), ("er", "e"), ("oi", "va"), ("w", "v"),
             ("ce", "se"), ("ci", "si"), ("cy", "si"), ("ge", "je"), ("gi", "ji"), ("c", "k")]


def _key(word, rules, strip_final=""):
    w = word
    for a, b in rules:
        w = w.replace(a, b)
    w = re.sub(r"(.)\1+", r"\1", w)
    if strip_final:
        w = re.sub(f"[{strip_final}]+$", "", w) or w
    return (w[0] + re.sub(r"[aeiou]", "", w[1:])) if w else ""


def phonetic_key(norm_text, style="indic"):
    """style="indic": sri/shri/shree/sree -> sr, lakshmi/laxmi -> lxm, chaudhary/chowdhury -> ctr.
    style="french": dupont/dupond -> dpn, thibault/thibaut -> tp. Words with digits pass through."""
    rules, strip = (_IND_RULES, "") if style == "indic" else (_FR_RULES, "stdxzpe")
    return " ".join(w if any(c.isdigit() for c in w) else _key(w, rules, strip) for w in norm_text.split())


# --------------------------------------------------------------------------------------
# Name / address structure
# --------------------------------------------------------------------------------------
_DBA = re.compile(r"\b(?:d\s*/?\s*b\s*/?\s*a|doing business as|t\s*/\s*a|trading as|a\s*/?\s*k\s*/?\s*a|"
                  r"formerly|f\s*/?\s*k\s*/?\s*a)\b\.?|[()\[\]]", re.I)
_LANDMARK = re.compile(r"^\s*(?:near|nr\.?|opp\.?|opposite|behind|beside|besides|next to|adj\.?|adjacent to|"
                       r"in front of|above|below|close to|off|pres de|près de|a cote de|à côté de|"
                       r"en face de|face a|face à|derriere|derrière)\b", re.I)
_HI_LANDMARK = re.compile(r"(?:\b\w+\s+){1,3}\bke\s+(?:paas|pass|saamne|samne|peeche|piche|bagal)\b", re.I)
_LANDMARK_INLINE = re.compile(r"\b(?:near|nr|opp|opposite|behind|beside|next to|in front of)\b(?:\s+[a-z]+){1,3}", re.I)


def split_dba(raw_name):
    """'ABC Holdings LLC dba Joe's Pizza' -> ['ABC Holdings LLC', "Joe's Pizza"]."""
    parts = [p.strip(" ,.-") for p in _DBA.split(raw_name or "")]
    return [p for p in parts if p] or [raw_name or ""]


def split_landmarks(raw_addr):
    """Returns (core_address, landmark_text) from the RAW address (commas still present)."""
    raw = raw_addr or ""
    marks = [m.group(0) for m in _HI_LANDMARK.finditer(raw)]
    raw = _HI_LANDMARK.sub(" ", raw)
    core = []
    for seg in re.split(r"[,;]", raw):
        (marks if _LANDMARK.match(seg) else core).append(seg)
    core_txt = ", ".join(core)
    if not marks:  # no comma around the landmark: remove keyword + up to 3 words
        marks = [m.group(0) for m in _LANDMARK_INLINE.finditer(core_txt)]
        core_txt = _LANDMARK_INLINE.sub(" ", core_txt)
    return core_txt, " ".join(marks)


_UNIT = re.compile(r"\b(?:ste|apt|unit|fl|shop|office|room|bldg|bureau|flat|wing|block|blk)\s*(?:no\s*)?#?\s*([a-z]?\d+[a-z]?)\b")
_PIN = re.compile(r"\b[1-9]\d{5}\b")
_ZIP = re.compile(r"\b\d{5}\b")
_HOUSE = re.compile(r"\b\d+[a-z]?(?:-\d+[a-z]?)*\b")
_ARR = {"75": "paris", "69": "lyon", "13": "marseille"}


def address_parts(norm_addr):
    """Structured pieces of a normalize_ml(kind="addr") string."""
    pc = (_PIN.findall(norm_addr) or _ZIP.findall(norm_addr) or [""])[-1]
    unit = _UNIT.search(norm_addr)
    no_unit = _UNIT.sub(" ", norm_addr)
    house = [h for h in _HOUSE.findall(no_unit) if h != pc]
    arr = ""
    if len(pc) == 5 and pc[:2] in _ARR and pc[2] == "0" and pc[3:] != "00":
        arr = f"{_ARR[pc[:2]]}{int(pc[3:])}"
    else:
        m = re.search(r"\b(paris|lyon|marseille) (\d{1,2})\b", norm_addr)
        arr = f"{m.group(1)}{int(m.group(2))}" if m else ""
    return {"pc": pc, "unit": unit.group(1) if unit else "", "house": house[0] if house else "", "arr": arr}


# --------------------------------------------------------------------------------------
# Abbreviation-aware soft token similarity (generalises to unseen languages)
# --------------------------------------------------------------------------------------
def _is_abbrev(short, long):
    """'mfg'<-'manufacturing', 'svcs'<-'services', 'bd'<-'boulevard', 'ste'<-'societe', 'r'<-'rajesh'."""
    if not short or short[0] != long[0] or len(short) >= len(long):
        return False
    it = iter(long)
    return all(c in it for c in short)


def _tok_sim(a, b):
    if a == b:
        return 1.0
    s, l = (a, b) if len(a) <= len(b) else (b, a)
    if not s.isdigit() and _is_abbrev(s, l):
        return 0.6 if len(s) == 1 else 0.85
    return JaroWinkler.normalized_similarity(a, b)


def soft_token_sim(x, y):
    """Monge-Elkan over tokens with abbreviation credit, both directions -> (min, max). NaN if a side is empty."""
    tx, ty = x.split(), y.split()
    if not tx or not ty:
        return np.nan, np.nan
    fwd = np.mean([max(_tok_sim(a, b) for b in ty) for a in tx])
    bwd = np.mean([max(_tok_sim(a, b) for a in tx) for b in ty])
    return float(min(fwd, bwd)), float(max(fwd, bwd))


# --------------------------------------------------------------------------------------
# Data-driven variant mining (train positives, or high-confidence address-anchored pairs)
# --------------------------------------------------------------------------------------
def mine_token_variants(pairs, min_count=10, max_side=3):
    """pairs: iterable of (norm_text_a, norm_text_b) for TRUE matches. Returns candidate
    token equivalences (tok_a, tok_b, n, conf) - review the top rows and add the good ones to
    _CANON. Finds transliteration variants (shree/sri), city renames, local abbreviations and
    cross-language words that actually occur in this dataset."""
    pair_n, a_n, b_n = Counter(), Counter(), Counter()
    for a, b in pairs:
        ta, tb = set(a.split()), set(b.split())
        oa, ob = ta - tb, tb - ta
        if not oa or not ob or len(oa) > max_side or len(ob) > max_side:
            continue
        a_n.update(oa)
        b_n.update(ob)
        pair_n.update((x, y) for x in oa for y in ob)
    rows = [(x, y, n, n / min(a_n[x], b_n[y])) for (x, y), n in pair_n.items() if n >= min_count]
    return (pd.DataFrame(rows, columns=["tok_a", "tok_b", "n", "conf"])
            .sort_values(["conf", "n"], ascending=False).reset_index(drop=True))


# --------------------------------------------------------------------------------------
# Pair features built on the above: normalise each RECORD once, then compare pairs
# --------------------------------------------------------------------------------------
def prepare_ml(df, name_col="business_name", addr_col="business_address"):
    """Per-record multilingual columns (run once per table; ~30 us per string)."""
    out = pd.DataFrame(index=df.index)
    names = df[name_col].fillna("").astype(str)
    addrs = df[addr_col].fillna("").astype(str)
    out["ml_name"] = names.map(lambda x: normalize_ml(x, "name"))
    out["ml_addr"] = addrs.map(lambda x: normalize_ml(x, "addr"))
    split = addrs.map(split_landmarks)
    out["ml_addr_core"] = split.map(lambda x: normalize_ml(x[0], "addr"))
    out["ml_landmark"] = split.map(lambda x: normalize_ml(x[1], "addr"))
    out["ml_dba"] = names.map(lambda x: tuple(normalize_ml(p, "name") for p in split_dba(x)))
    out["key_indic"] = out.ml_name.map(lambda x: phonetic_key(x, "indic"))
    out["key_french"] = out.ml_name.map(lambda x: phonetic_key(x, "french"))
    parts = out.ml_addr.map(address_parts)
    for k in ("pc", "unit", "house", "arr"):
        out[f"ml_{k}"] = parts.map(lambda d, k=k: d[k])
    out["ml_name_nums"] = out.ml_name.map(lambda x: frozenset(re.findall(r"\d+", x)))
    out["nonlatin"] = names.map(lambda x: any(script_of(c) not in ("ascii", "latin_accented") for c in x if c.isalpha()))
    return out


def ml_pair_features(qi, di, A, B):
    """qi/di: integer row positions into prepared tables A (Source 1) and B (Source 2/3)."""
    g = lambda T, c, idx: T[c].to_numpy()[idx]
    an, bn = g(A, "ml_name", qi), g(B, "ml_name", di)
    ac, bc = g(A, "ml_addr_core", qi), g(B, "ml_addr_core", di)
    F = pd.DataFrame(index=range(len(qi)))
    soft_n = [soft_token_sim(x, y) for x, y in zip(an, bn)]
    soft_a = [soft_token_sim(x, y) for x, y in zip(ac, bc)]
    F["name_soft_min"], F["name_soft_max"] = zip(*soft_n) if len(qi) else ([], [])
    F["addr_soft_min"], F["addr_soft_max"] = zip(*soft_a) if len(qi) else ([], [])
    ts = lambda xs, ys: [np.nan if not x or not y else fuzz.token_set_ratio(x, y) / 100 for x, y in zip(xs, ys)]
    F["name_tset_ml"] = ts(an, bn)
    F["addr_core_tset"] = ts(ac, bc)
    F["name_key_indic"] = ts(g(A, "key_indic", qi), g(B, "key_indic", di))
    F["name_key_french"] = ts(g(A, "key_french", qi), g(B, "key_french", di))
    la, lb = g(A, "ml_landmark", qi), g(B, "ml_landmark", di)
    F["landmark_a"], F["landmark_b"] = (la != "").astype(float), (lb != "").astype(float)
    F["landmark_sim"] = ts(la, lb)
    da, db = g(A, "ml_dba", qi), g(B, "ml_dba", di)
    F["has_dba"] = [float(len(x) > 1 or len(y) > 1) for x, y in zip(da, db)]
    F["name_dba_best"] = [max((fuzz.token_set_ratio(p, q) / 100 for p in x for q in y if p and q), default=np.nan)
                          if (len(x) > 1 or len(y) > 1) else np.nan for x, y in zip(da, db)]
    na, nb = g(A, "ml_name_nums", qi), g(B, "ml_name_nums", di)
    F["name_num_conflict"] = [float(bool(x and y and x != y)) for x, y in zip(na, nb)]
    for k in ("pc", "unit", "house", "arr"):
        x, y = g(A, f"ml_{k}", qi), g(B, f"ml_{k}", di)
        F[f"{k}_eq"] = np.where((x != "") & (y != ""), (x == y).astype(float), np.nan)
    xa, xb = g(A, "nonlatin", qi).astype(float), g(B, "nonlatin", di).astype(float)
    F["nonlatin_a"], F["nonlatin_b"], F["cross_script"] = xa, xb, (xa != xb).astype(float)
    return F

In [ ]:
"""
er_lexicon.py - static preprocessing resources (built once offline; see jev/README.md).

No API is called here. The resources are plain TSV files in src/resources/ (or, in the Kaggle
notebook, the same files embedded as a compressed string):
  lexicon.tsv         country, field, variant, canonical   word/phrase equivalences per country
  token_class.tsv     country, field, tok, kind, p         legal_form / title / generic_business ...
  conflict_pairs.tsv  country, field, a, b, source          look-alike words that are different names

What it adds on top of er_multilingual.normalize_ml:
  1. OCR-noise repair in names (hea1th -> health, 6roup -> group, lnc -> inc)
  2. per-country phrase rules and token map (tamil nadu -> tn, r -> rue, kansaltents -> consultants)
  3. junk address tokens dropped (null, na, pmb)
and two pair signals: distinct_tokens (name without legal/generic words) and name_conflict.
"""
import base64
import gzip
import io
import json
import os
import re

import pandas as pd

ADDR_DROP = {"null", "na", "nan", "none", "pmb"}  # literal junk tokens seen in US/India addresses
NON_DISTINCT = {"legal_form", "title", "connector", "generic_business"}
_OCR = str.maketrans({"0": "o", "1": "l", "5": "s", "6": "g", "8": "b"})
_OCR_TOKEN = re.compile(r"[A-Za-z]*[01568][A-Za-z01568]*")
_L_FOR_I = re.compile(r"\bl(?=[nm][a-z]+)")
_EMPTY = {"tok": {"name": {}, "addr": {}}, "phr": {"name": [], "addr": []}}


def fix_ocr(text):
    """Digit-for-letter noise inside words. Only tokens with <=2 digits, all from {0,1,5,6,8}, and
    >=3 letters (or 2 letters around one inner digit: c0m) change, so house numbers, PINs and codes
    (12b, a18a, 4x4, A1) are untouched."""
    def rep(m):
        w = m.group(0)
        digits = sum(c.isdigit() for c in w)
        letters = len(w) - digits
        inner = len(w) >= 3 and w[0].isalpha() and w[-1].isalpha()
        if digits == 0 or digits > 2 or not (letters >= 3 or (letters == 2 and digits == 1 and inner)):
            return w
        return w.translate(_OCR)
    t = _OCR_TOKEN.sub(rep, text)
    return _L_FOR_I.sub("i", t.lower()) if t else t


def country_key(c):
    """Unknown countries fall back to the '*' (script-level) lexicon, so an unseen country works."""
    s = re.sub(r"[^a-z]", "", str(c).lower())
    return {"us": "US", "usa": "US", "unitedstates": "US", "unitedstatesofamerica": "US",
            "in": "INDIA", "ind": "INDIA", "india": "INDIA", "bharat": "INDIA",
            "fr": "FRANCE", "fra": "FRANCE", "france": "FRANCE"}.get(s, "*")


def _from_frames(lexicon=None, token_class=None, conflict_pairs=None):
    lex = {}
    if lexicon is not None:
        for c, f, v, t in zip(lexicon.country, lexicon.field, lexicon.variant, lexicon.canonical):
            L = lex.setdefault(c, {"tok": {"name": {}, "addr": {}}, "phr": {"name": [], "addr": []}})
            for ff in (("name", "addr") if f == "both" else (f,)):
                (L["phr"][ff].append((v, t)) if " " in v else L["tok"][ff].__setitem__(v, t))
    lex.setdefault("*", {"tok": {"name": {}, "addr": {}}, "phr": {"name": [], "addr": []}})
    tc = {} if token_class is None else {(c, f, t): k for c, f, t, k in zip(
        token_class.country, token_class.field, token_class.tok, token_class.kind)}
    cf = set() if conflict_pairs is None else {(c, f, a, b) for c, f, a, b in zip(
        conflict_pairs.country, conflict_pairs.field, conflict_pairs.a, conflict_pairs.b)}
    return {"lex": lex, "token_class": tc, "conflicts": cf}


_FILES = ("lexicon", "token_class", "conflict_pairs")


def _read_tsv_text(txt):
    return pd.read_csv(io.StringIO(txt), sep="\t", dtype=str, keep_default_na=False)


def load(folder):
    frames = {}
    for n in _FILES:
        p = os.path.join(folder, f"{n}.tsv")
        if os.path.exists(p):
            frames[n] = pd.read_csv(p, sep="\t", dtype=str, keep_default_na=False)
    return _from_frames(**frames)


def load_default():
    return load(os.path.join(os.path.dirname(os.path.abspath(__file__)), "resources"))


def embed(folder):
    """-> compact ASCII string holding the resource files (used by the notebook generator)."""
    payload = {n: open(os.path.join(folder, f"{n}.tsv"), encoding="utf-8").read()
               for n in _FILES if os.path.exists(os.path.join(folder, f"{n}.tsv"))}
    return base64.b64encode(gzip.compress(json.dumps(payload).encode("utf-8"), 9, mtime=0)).decode("ascii")


def load_embedded(blob):
    payload = json.loads(gzip.decompress(base64.b64decode(blob)).decode("utf-8"))
    return _from_frames(**{n: _read_tsv_text(t) for n, t in payload.items()})


def wrap(normalize_ml, LEX, country="*"):
    """normalize_ml'(text, kind) for one country. Hand-written _CANON rules always win: a lexicon
    entry whose variant normalize_ml already rewrites is ignored. Phrase keys and targets are
    normalised with the ORIGINAL normalize_ml so they live in its canonical space."""
    L = LEX["lex"].get(country) or LEX["lex"].get("*") or _EMPTY
    compiled = {}
    for kind in ("name", "addr"):
        rules = []
        for v, c in L["phr"][kind]:
            key = normalize_ml(v, kind)
            if key and " " in key:
                rules.append((re.compile(rf"(?<!\S){re.escape(key)}(?!\S)"), normalize_ml(c, kind) or c))
        rules.sort(key=lambda r: -len(r[0].pattern))  # longest phrase first
        tmap = {}
        for v, c in L["tok"][kind].items():
            tv, tc = normalize_ml(v, kind), normalize_ml(c, kind) or c
            if tv != v:
                continue
            if tv and " " not in tv and tv != tc:
                tmap[tv] = tc
        compiled[kind] = (rules, tmap)

    def normalize_ml2(text, kind="name"):
        if kind == "name" and isinstance(text, str):
            text = fix_ocr(text)
        t = normalize_ml(text, kind)
        if not t:
            return t
        rules, tmap = compiled.get(kind, ([], {}))
        for rx, rep in rules:
            t = rx.sub(rep, t)
        toks = (tmap.get(w, w) for w in t.split())
        if kind == "addr":
            toks = (w for w in toks if w not in ADDR_DROP)
        return " ".join(toks)

    normalize_ml2.country = country
    return normalize_ml2


def country_aware(prepare_ml, namespace, LEX, country_col="country"):
    """Wraps prepare_ml(df, ...) so each country's rows are normalised with that country's lexicon.
    `namespace` is the dict where prepare_ml looks up normalize_ml (notebook: globals();
    package: vars(er_multilingual)). Row order and index are preserved; the original
    normalize_ml is always restored."""
    base = namespace["normalize_ml"]
    norms = {c: wrap(base, LEX, c) for c in LEX["lex"]}

    def prepare_ml2(df, *args, **kwargs):
        if country_col not in df:
            return prepare_ml(df, *args, **kwargs)
        keys = df[country_col].map(country_key)
        parts = []
        try:
            for c, idx in keys.groupby(keys).groups.items():
                namespace["normalize_ml"] = norms.get(c, norms["*"])
                parts.append(prepare_ml(df.loc[idx], *args, **kwargs))
        finally:
            namespace["normalize_ml"] = base
        return pd.concat(parts).loc[df.index] if parts else prepare_ml(df, *args, **kwargs)

    prepare_ml2.lexicon_wrapped = True
    return prepare_ml2


def distinct_tokens(norm_name, LEX, country="*"):
    """Tokens that identify the business: drops legal forms, titles, connectors and generic business
    words ('services', 'groupe', 'traders'). Falls back to all tokens."""
    tc = LEX["token_class"]
    toks = str(norm_name).split()
    keep = [t for t in toks if tc.get((country, "name", t)) not in NON_DISTINCT]
    return keep or toks


def name_conflict(a_tokens, b_tokens, LEX, country="*", field="name"):
    """1.0 if the names differ by a known look-alike pair of different names (patel vs patil)."""
    A, B = set(a_tokens) - set(b_tokens), set(b_tokens) - set(a_tokens)
    if not A or not B:
        return 0.0
    C = LEX["conflicts"]
    return float(any((country, field, min(x, y), max(x, y)) in C for x in A for y in B))


# ---- resources embedded by the generator (same content as src/resources/*.tsv) ----
_LEX_BLOB = "..."
_LEX = load_embedded(_LEX_BLOB)
if not getattr(prepare_ml, "lexicon_wrapped", False):
    prepare_ml = country_aware(prepare_ml, globals(), _LEX)
print("Lexicon loaded:", {c: len(v["tok"]["name"]) + len(v["tok"]["addr"]) + len(v["phr"]["addr"]) for c, v in _LEX["lex"].items()},
      "| token classes:", len(_LEX["token_class"]), "| conflict pairs:", len(_LEX["conflicts"]))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

def _topk_rows(S, k, min_sim):
    """Extract top-k column indices and similarity scores per CSR row."""
    rows, cols, vals = [], [], []
    ip, ix, dv = S.indptr, S.indices, S.data
    for r in range(S.shape[0]):
        a, b = ip[r], ip[r + 1]
        if a == b:
            continue
        seg = dv[a:b]
        if b - a > k:
            sel = np.argpartition(-seg, k)[:k]
        else:
            sel = np.arange(b - a)
        sel = sel[seg[sel] >= min_sim]
        rows.append(np.full(len(sel), r, dtype=np.int32))
        cols.append(ix[a:b][sel])
        vals.append(seg[sel])
    if not rows:
        return np.array([], np.int32), np.array([], np.int32), np.array([], np.float32)
    return np.concatenate(rows), np.concatenate(cols), np.concatenate(vals)

def block_topk(q_text, d_text, q_country, d_country, k=25, min_sim=0.01,
               max_df=0.01, ngram_range=(3, 4), batch=2000, fit_on="both"):
    """Scalable character n-gram TF-IDF blocking partitioned by country.
    
    1. Splits search space by country (US, INDIA, FRANCE) to eliminate cross-country waste.
    2. Drops frequent n-grams with max_df=0.01 to drop sparse matrix density from ~90% to ~5-7%.
    3. Transposes target matrix once.
    4. Returns DataFrame with integer row indices (qi, di) and similarity.
    """
    q_text, d_text = np.asarray(q_text, dtype=object), np.asarray(d_text, dtype=object)
    q_country, d_country = np.asarray(q_country, dtype=object), np.asarray(d_country, dtype=object)
    out = []
    
    for g in pd.unique(q_country):
        qi = np.flatnonzero(q_country == g)
        di = np.arange(len(d_text)) if g == "" else np.flatnonzero((d_country == g) | (d_country == ""))
        if len(qi) == 0 or len(di) == 0:
            continue
            
        vec = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=ngram_range,
            min_df=2,
            max_df=max_df,
            sublinear_tf=True,
            dtype=np.float32
        )
        corpus = np.concatenate([q_text[qi], d_text[di]]) if fit_on == "both" else d_text[di]
        vec.fit(corpus)
        
        Q = vec.transform(q_text[qi])
        DT = vec.transform(d_text[di]).T.tocsr()
        kk = min(k, len(di))
        
        for s in range(0, len(qi), batch):
            S = (Q[s:s + batch] @ DT).tocsr()
            r, c, v = _topk_rows(S, kk, min_sim)
            out.append(pd.DataFrame({"qi": qi[s + r], "di": di[c], "sim": v}))
            
    if not out:
        return pd.DataFrame({"qi": [], "di": [], "sim": []})
    return pd.concat(out, ignore_index=True)

def union_candidates(named_blocks):
    """Combines multiple blocker outputs (e.g. name, text combo) on [qi, di]."""
    merged = None
    for nm, df in named_blocks.items():
        if df.empty:
            continue
        df_named = df.rename(columns={"sim": f"blk_{nm}"})
        merged = df_named if merged is None else merged.merge(df_named, on=["qi", "di"], how="outer")
        
    if merged is None or merged.empty:
        return pd.DataFrame(columns=["qi", "di", "n_blockers"])
        
    blk_cols = [c for c in merged.columns if c.startswith("blk_")]
    merged["n_blockers"] = merged[blk_cols].notna().sum(axis=1)
    return merged

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
from sklearn.feature_extraction.text import TfidfVectorizer


_PC = re.compile(r"\d{5,6}")
_NUM = re.compile(r"\d+")

def _first(rx, s):
    m = rx.search(str(s))
    return m.group(0) if m else ""

def _rowwise_cos(A, B, ia, ib, chunk=500_000):
    out = np.empty(len(ia), dtype=np.float32)
    for s in range(0, len(ia), chunk):
        out[s:s + chunk] = np.asarray(A[ia[s:s + chunk]].multiply(B[ib[s:s + chunk]]).sum(axis=1)).ravel()
    return out

_TFIDF_CACHE = {}


def _pool_tfidf(s23, col):
    key = (id(s23), len(s23), col, str(s23["entity_id"].iat[0]) if len(s23) else "")
    if key not in _TFIDF_CACHE:
        if len(_TFIDF_CACHE) > 8:
            _TFIDF_CACHE.clear()
        v = TfidfVectorizer(analyzer="word", token_pattern=r"\S+", sublinear_tf=True, dtype=np.float32)
        B = v.fit_transform(s23[col].to_numpy())
        _TFIDF_CACHE[key] = (v, B.tocsr())
    return _TFIDF_CACHE[key]


def _idfcos(cand, s1, s23):
    qi, di = cand.qi.to_numpy(), cand.di.to_numpy()
    out = {}
    for col, key in (("norm_name", "name"), ("norm_addr", "addr")):
        v, B = _pool_tfidf(s23, col)
        out[f"{key}_idfcos"] = _rowwise_cos(v.transform(s1[col]), B, qi, di)
    return out


def _pf_task(args):
    c, a, b = args
    return _pair_features(c, a, b, workers=1, idfcos=False)


def pair_features(cand, s1, s23, workers=-1, n_jobs=None):
    """Pairwise features. Large inputs are split across CPU cores; each worker receives only its pairs
    and the Source 1 / Source 2/3 rows they reference (re-indexed), so memory stays bounded. The IDF
    cosines need the full Source 2/3 pool and are computed in the parent."""
    n = n_jobs or N_JOBS
    if n <= 1 or len(cand) < 200_000:
        return _pair_features(cand, s1, s23, workers)
    order = list(_pair_features(cand.iloc[:200], s1, s23, workers).columns)
    keep1 = [c for c in s1.columns if c not in _DROP_FOR_WORKERS]
    keep2 = [c for c in s23.columns if c not in _DROP_FOR_WORKERS]
    n_parts = max(n * 3, int(np.ceil(len(cand) / MAX_PAIRS_PER_TASK)))  # small tasks bound worker memory
    parts = np.array_split(np.arange(len(cand)), n_parts)

    def make(i):
        c = cand.iloc[parts[i]].copy()
        qu, qinv = np.unique(c["qi"].to_numpy(), return_inverse=True)
        du, dinv = np.unique(c["di"].to_numpy(), return_inverse=True)
        c["qi"], c["di"] = qinv, dinv
        return c, s1.iloc[qu][keep1].reset_index(drop=True), s23.iloc[du][keep2].reset_index(drop=True)

    tasks = _Lazy(len(parts), make)  # built one at a time as workers ask for them
    F = pd.concat(_fork_map(_pf_task, tasks, n))
    del tasks
    for key, arr in _idfcos(cand, s1, s23).items():
        F[key] = arr
    return F[order]


def _jacc(x, y):
    a, b = set(x.split()), set(y.split())
    return len(a & b) / len(a | b) if (a or b) else np.nan


N_JOBS = int(os.environ.get("ER_N_JOBS", "0")) or (os.cpu_count() or 1)
MAX_PAIRS_PER_TASK = 100_000


class _Lazy:
    """Sequence whose items are built on demand (keeps only in-flight tasks in memory)."""

    def __init__(self, n, make):
        self.n, self.make = n, make

    def __len__(self):
        return self.n

    def __iter__(self):
        return (self.make(i) for i in range(self.n))
_DROP_FOR_WORKERS = ("business_name", "business_address", "ml_addr")


def _fork_map(fn, items, n):
    """Ordered map over n forked worker processes. `fn` is inherited through fork (works for notebook-
    defined functions); each task is PICKLED to the worker through a bounded queue, so workers only touch
    their own private copy and never the parent's objects (reading an inherited Python object writes its
    reference count, which would copy that memory page into every worker). Results come back pickled.
    Raises instead of hanging if a worker fails or is killed (e.g. out of memory)."""
    import gc as _gc
    import multiprocessing as mp
    import queue as _queue
    import threading
    import traceback
    ctx = mp.get_context("fork")
    tasks, results = ctx.Queue(maxsize=max(2, n)), ctx.Queue()

    def work():
        while True:
            job = tasks.get()
            if job is None:
                return
            i, payload = job
            del job
            try:
                results.put((i, True, fn(payload)))
            except BaseException:
                results.put((i, False, traceback.format_exc()))
                return
            del payload

    _gc.collect()
    _gc.freeze()
    procs = [ctx.Process(target=work, daemon=True) for _ in range(min(n, len(items)))]
    for p in procs:
        p.start()
    _gc.unfreeze()
    stop = threading.Event()

    def feed():
        for i, it in enumerate(items):
            while not stop.is_set():
                try:
                    tasks.put((i, it), timeout=1)
                    break
                except _queue.Full:
                    continue
        for _ in procs:
            while not stop.is_set():
                try:
                    tasks.put(None, timeout=1)
                    break
                except _queue.Full:
                    continue

    feeder = threading.Thread(target=feed, daemon=True)
    feeder.start()
    try:
        out, done = [None] * len(items), 0
        while done < len(items):
            try:
                i, ok, val = results.get(timeout=5)
            except _queue.Empty:
                dead = [p for p in procs if not p.is_alive() and p.exitcode not in (0, None)]
                if dead:
                    raise RuntimeError(f"worker died (exit code {dead[0].exitcode}; -9 usually means out of memory)")
                continue
            if not ok:
                raise RuntimeError(f"worker failed:\n{val}")
            out[i] = val
            done += 1
        return out
    finally:
        stop.set()
        for p in procs:
            p.join(timeout=5)
            if p.is_alive():
                p.terminate()
        feeder.join(timeout=5)


def _prepare_rows(df, drop=()):
    """All per-row work of prepare_side (everything except the table-level core frequency)."""
    df = df.copy()
    ml_cols = prepare_ml(df, name_col="business_name", addr_col="business_address")
    for col in ml_cols.columns:
        df[col] = ml_cols[col]
    df["norm_name"] = df["ml_name"]
    df["norm_addr"] = df["ml_addr_core"]
    df["country"] = df["country"].map(canon_country) if "country" in df.columns else ""
    df["core"] = df["norm_name"].map(core_name)
    df["pc"] = df["ml_pc"]
    df["house"] = df["ml_house"]
    df["nums"] = df["norm_addr"].map(lambda s: frozenset(_NUM.findall(str(s))))
    df["ckey"] = df["country"].map(country_key)
    df["distinct"] = [" ".join(distinct_tokens(n, _LEX, c)) for n, c in zip(df["norm_name"], df["ckey"])]
    _share_objects(df)
    return df.drop(columns=list(drop), errors="ignore")


def _share_objects(df):
    """Same values, less memory (~550 bytes/row on 10M rows): identical number-sets become one shared
    object (97% of name number-sets are empty), and core / distinct / DBA parts equal to the normalised
    name reuse that string instead of holding a copy."""
    for col in ("nums", "ml_name_nums"):
        if col in df:
            cache = {}
            df[col] = [cache.setdefault(v, v) for v in df[col]]
    name = df["norm_name"].to_numpy(object)
    for col in ("core", "distinct"):
        df[col] = [n if v == n else v for v, n in zip(df[col].to_numpy(object), name)]
    if "ml_dba" in df:
        df["ml_dba"] = [(n,) if len(t) == 1 and t[0] == n else t for t, n in zip(df["ml_dba"], name)]


def _prepare_rows_task(args):
    return _prepare_rows(*args)


def prepare_side(df, drop=(), n_jobs=None):
    """Precomputes side-level metadata, multilingual representations, and chain frequencies.
    Runs on all CPU cores for large tables. `drop` removes raw columns afterwards (saves memory)."""
    n = n_jobs or N_JOBS
    if n > 1 and len(df) >= 40_000:
        parts = np.array_split(np.arange(len(df)), n * 4)
        out = pd.concat(_fork_map(_prepare_rows_task, _Lazy(len(parts), lambda i: (df.iloc[parts[i]], tuple(drop))), n))
        for col in ("nums", "ml_name_nums"):  # share identical sets across worker parts too
            if col in out:
                cache = {}
                out[col] = [cache.setdefault(v, v) for v in out[col]]
    else:
        out = _prepare_rows(df, drop)
    out["core_freq"] = out.groupby(["country", "core"])["core"].transform("size").astype(np.float32)
    return out


def _pair_features(cand, s1, s23, workers=-1, idfcos=True):
    """Computes unified 51-dimensional pairwise features across name, address, postcodes, and multilingual representations."""
    qi, di = cand.qi.to_numpy(), cand.di.to_numpy()
    F = pd.DataFrame(index=cand.index)
    
    blk_cols = [c for c in cand.columns if c.startswith("blk_")]
    if "n_blockers" in cand:
        blk_cols.append("n_blockers")
    for c in blk_cols:
        F[c] = cand[c].to_numpy(np.float32)

    an, bn = s1.norm_name.to_numpy()[qi], s23.norm_name.to_numpy()[di]
    ac, bc = s1.core.to_numpy()[qi], s23.core.to_numpy()[di]
    aa, ba = s1.norm_addr.to_numpy()[qi], s23.norm_addr.to_numpy()[di]

    scorers = {
        "tset": fuzz.token_set_ratio,
        "tsort": fuzz.token_sort_ratio,
        "part": fuzz.partial_ratio,
        "ratio": fuzz.ratio,
        "wr": fuzz.WRatio
    }
    for nm, sc in scorers.items():
        F[f"name_{nm}"] = cpdist(an, bn, scorer=sc, workers=workers, dtype=np.float32) / 100.0
        F[f"core_{nm}"] = cpdist(ac, bc, scorer=sc, workers=workers, dtype=np.float32) / 100.0
        
    F["name_jw"] = cpdist(an, bn, scorer=JaroWinkler.normalized_similarity, workers=workers, dtype=np.float32)
    F["core_exact"] = (ac == bc).astype(np.float32)
    
    for nm in ("tset", "part", "ratio"):
        F[f"addr_{nm}"] = cpdist(aa, ba, scorer=scorers[nm], workers=workers, dtype=np.float32) / 100.0

    # Rare token IDF-weighted word cosine (vectoriser fitted once per S2/S3 pool and cached)
    if idfcos:
        for key, arr in _idfcos(cand, s1, s23).items():
            F[key] = arr

    pa, pb = s1.pc.to_numpy()[qi], s23.pc.to_numpy()[di]
    both = (pa != "") & (pb != "")
    F["pc_both"] = both.astype(np.float32)
    F["pc_eq"] = np.where(both, (pa == pb).astype(np.float32), np.nan)
    
    ha, hb = s1.house.to_numpy()[qi], s23.house.to_numpy()[di]
    F["house_eq"] = np.where((ha != "") & (hb != ""), (ha == hb).astype(np.float32), np.nan)
    
    na, nb = s1.nums.to_numpy()[qi], s23.nums.to_numpy()[di]
    F["num_jacc"] = np.array([len(x & y) / len(x | y) if (x or y) else np.nan for x, y in zip(na, nb)], np.float32)

    ia = np.array([_initials(x) for x in ac], dtype=object)
    ib = np.array([_initials(x) for x in bc], dtype=object)
    F["acronym"] = (((ia != "") & (ia == np.char.replace(bc.astype(str), " ", ""))) |
                    ((ib != "") & (ib == np.char.replace(ac.astype(str), " ", "")))).astype(np.float32)

    F["core_freq_s1"] = np.log1p(s1.core_freq.to_numpy()[qi])
    F["core_freq_s23"] = np.log1p(s23.core_freq.to_numpy()[di])
    
    la, lb = np.char.str_len(an.astype(str)), np.char.str_len(bn.astype(str))
    F["name_len_ratio"] = (np.minimum(la, lb) / np.maximum(np.maximum(la, lb), 1)).astype(np.float32)
    F["addr_len_a"] = np.char.str_len(aa.astype(str)).astype(np.float32)
    F["addr_len_b"] = np.char.str_len(ba.astype(str)).astype(np.float32)
    F["is_s3"] = np.char.startswith(s23.entity_id.to_numpy()[di].astype(str), "S3-").astype(np.float32)

    # Distinctive-word name similarity (legal forms, titles, generic business words removed) and
    # look-alike conflict (the names differ by a known pair of different names: patel / patil)
    xa, xb = s1["distinct"].to_numpy()[qi], s23["distinct"].to_numpy()[di]
    F["distinct_tset"] = cpdist(xa, xb, scorer=fuzz.token_set_ratio, workers=workers, dtype=np.float32) / 100.0
    F["distinct_jacc"] = np.array([_jacc(x, y) for x, y in zip(xa, xb)], np.float32)
    ck = s1["ckey"].to_numpy()[qi]
    F["name_conflict"] = np.array([name_conflict(x.split(), y.split(), _LEX, c) for x, y, c in zip(an, bn, ck)],
                                  np.float32)

    # Multilingual structured signals (abbreviations, landmarks, DBAs, non-Latin flags)
    F_ml = ml_pair_features(qi, di, s1, s23)
    for col in F_ml.columns:
        if col not in F.columns:
            F[col] = F_ml[col].to_numpy(np.float32)

    return F


def _side_stats(key, score):
    order = np.lexsort((-score, key))
    k_sorted, s_sorted = key[order], score[order]
    new = np.r_[True, k_sorted[1:] != k_sorted[:-1]]
    gid = np.cumsum(new) - 1
    starts = np.flatnonzero(new)
    sizes = np.diff(np.r_[starts, len(key)])
    rank = np.arange(len(key)) - starts[gid] + 1
    top1 = s_sorted[starts]
    top2 = np.where(sizes > 1, s_sorted[np.minimum(starts + 1, len(key) - 1)], np.nan)
    best_other = np.where(rank == 1, top2[gid], top1[gid])
    out_rank, out_size, out_margin = (np.empty(len(key)) for _ in range(3))
    out_rank[order], out_size[order] = rank, sizes[gid]
    out_margin[order] = s_sorted - best_other
    return out_rank, out_size, out_margin

def add_group_features(df, score_col="p1"):
    s = df[score_col].to_numpy(np.float64)
    for side, key in (("s1", df.qi.to_numpy()), ("s23", df.di.to_numpy())):
        r, n, m = _side_stats(key, s)
        df[f"{score_col}_rank_{side}"] = r
        df[f"{score_col}_n_{side}"] = n
        df[f"{score_col}_margin_{side}"] = m
    return df

In [ ]:
import numpy as np
import pandas as pd

def parse_gt(gt):
    """Parses ground truth into {s1_id: set_of_matched_ids}."""
    return {
        a: {x.strip() for x in m.split(",") if x.strip()}
        for a, m in zip(gt.iloc[:, 0], gt.iloc[:, 1].fillna(""))
    }

def gt_diagnostics(gt_dict, s1, s23):
    """Computes dataset structural statistics."""
    pairs = [(a, b) for a, ms in gt_dict.items() for b in ms]
    claims = pd.Series([b for _, b in pairs], dtype=object).value_counts()
    c1 = dict(zip(s1.entity_id, s1.country))
    c23 = dict(zip(s23.entity_id, s23.country))
    n_match = pd.Series([len(gt_dict.get(e, ())) for e in s1.entity_id], index=s1.country.values)
    return {
        "s1_total": len(s1),
        "s23_total": len(s23),
        "singleton_share": float((n_match == 0).mean()),
        "singleton_share_by_country": (n_match == 0).groupby(level=0).mean().round(4).to_dict(),
        "matches_per_s1_hist": n_match.value_counts().sort_index().head(8).to_dict(),
        "s23_claimed_by_more_than_one_s1": int((claims > 1).sum()),
        "cross_country_true_pairs": int(sum(c1.get(a) != c23.get(b) for a, b in pairs)),
        "s23_never_matched_share": float(1 - len(claims) / max(len(s23), 1)),
        "gt_ids_missing_from_sources": int(sum(b not in c23 for _, b in pairs)),
    }

def macro_f05(pred, n_true, s1_ids):
    """Vectorized calculation of the official Entity-Level Macro F0.5.
    
    Includes singleton credit (f=1.0 for true singletons with 0 predictions).
    """
    agg = pred.groupby("qi")["y"].agg(["size", "sum"]) if not pred.empty else pd.DataFrame(columns=["size", "sum"])
    npred = agg["size"].reindex(s1_ids, fill_value=0).to_numpy(float)
    tp = agg["sum"].reindex(s1_ids, fill_value=0).to_numpy(float)
    nt = np.asarray(n_true, float)[s1_ids]
    
    p = np.divide(tp, npred, out=np.zeros_like(tp), where=npred > 0)
    r = np.divide(tp, nt, out=np.zeros_like(tp), where=nt > 0)
    den = 0.25 * p + r
    f = np.divide(1.25 * p * r, den, out=np.zeros_like(tp), where=den > 0)
    f[(nt == 0) & (npred == 0)] = 1.0
    return float(f.mean())

def select_matches(df, score_col, tau1, tau2, one_to_one=True):
    """Two-threshold policy with one-to-one S2/S3 assignment.
    
    - tau1: threshold for each S1's top candidate
    - tau2: threshold (>= tau1) for additional candidates
    - one_to_one: ensures each S2/S3 record is claimed only by the S1 scoring it highest
    """
    if df.empty:
        return df
    d = df
    if one_to_one:
        d = d.loc[d.groupby("di")[score_col].idxmax()]
    d = d[d[score_col] >= min(tau1, tau2)]
    if d.empty:
        return d
    rank = d.groupby("qi")[score_col].rank(ascending=False, method="first")
    keep = ((rank == 1) & (d[score_col] >= tau1)) | ((rank > 1) & (d[score_col] >= tau2))
    return d[keep]

def tune_policy(df, score_col, n_true, s1_ids, one_to_one=True,
                grid1=np.arange(0.20, 0.81, 0.05), grid2=np.arange(0.30, 0.96, 0.05)):
    """Grid searches optimal (tau1, tau2) on validation data using exact Macro F0.5."""
    best = (-1.0, 0.5, 0.6)
    for t1 in grid1:
        for t2 in grid2:
            if t2 < t1:
                continue
            sel = select_matches(df, score_col, t1, t2, one_to_one)
            f = macro_f05(sel, n_true, s1_ids)
            if f > best[0]:
                best = (f, float(t1), float(t2))
    return best

In [ ]:
"""
groups.py - second-stage (group-context) features for the pair classifier.

A Source 1 entity usually has 3-4 true matches that also resemble EACH OTHER. Stage 1 scores every
pair alone, so a match whose name was replaced by a trade name ("Kororbikor" at the exact same
address as three confirmed matches) scores low. Stage 2 sees:
  * rank / count / margin of the stage-1 score within the Source 1 group and within the Source 2/3
    competitors (one-to-one structure),
  * how many strong matches the Source 1 entity already has,
  * similarity of the candidate to the best OTHER candidate of the same Source 1 (name, address,
    exact address, postcode/house agreement).
"""
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist


def _side_stats(key, score):
    order = np.lexsort((-score, key))
    k_sorted, s_sorted = key[order], score[order]
    new = np.r_[True, k_sorted[1:] != k_sorted[:-1]]
    gid = np.cumsum(new) - 1
    starts = np.flatnonzero(new)
    sizes = np.diff(np.r_[starts, len(key)])
    rank = np.arange(len(key)) - starts[gid] + 1
    top1 = s_sorted[starts]
    top2 = np.where(sizes > 1, s_sorted[np.minimum(starts + 1, len(key) - 1)], np.nan)
    best_other = np.where(rank == 1, top2[gid], top1[gid])
    r, n, m, bo = (np.empty(len(key)) for _ in range(4))
    r[order], n[order] = rank, sizes[gid]
    m[order] = s_sorted - best_other
    bo[order] = best_other
    return r, n, m, bo


def s1_side_features(cands, p_col, s23, strong=(0.5, 0.9)):
    """Features that only need the candidates of each Source 1 (safe to compute per S1 chunk).
    cands: qi, di, p_col. s23: prepared Source 2/3 table (norm_name, norm_addr, pc, house)."""
    qi, di = cands["qi"].to_numpy(), cands["di"].to_numpy()
    p = cands[p_col].to_numpy(np.float64)
    G = pd.DataFrame(index=cands.index)
    r, n, m, bo = _side_stats(qi, p)
    G["g_rank_s1"], G["g_n_s1"], G["g_margin_s1"], G["g_best_other_s1"] = r, n, m, bo
    for t in strong:
        cnt = pd.Series(p >= t).groupby(qi).transform("sum").to_numpy()
        G[f"g_n_ge{int(t * 100)}_s1"] = cnt - (p >= t)  # strong OTHER candidates
    # best OTHER candidate of the same Source 1: top-1, or top-2 for the top-1 row itself
    order = np.lexsort((-p, qi))
    q_sorted = qi[order]
    first = np.r_[True, q_sorted[1:] != q_sorted[:-1]]
    top1_pos = order[first]
    grp_start = np.cumsum(first) - 1
    top1_of = np.empty(len(qi), np.int64)
    top1_of[order] = top1_pos[grp_start]
    second = np.full(len(top1_pos), -1)
    idx_second = np.flatnonzero(first) + 1
    ok = (idx_second < len(order)) & ~np.r_[first, True][idx_second]
    second[ok] = order[idx_second[ok]]
    second_of = np.empty(len(qi), np.int64)
    second_of[order] = second[grp_start]
    other = np.where(np.arange(len(qi)) == top1_of, second_of, top1_of)
    has_other = other >= 0
    o_di = np.where(has_other, di[np.maximum(other, 0)], di)
    nn, na = s23["norm_name"].to_numpy(), s23["norm_addr"].to_numpy()
    G["g_top_name_tset"] = np.where(has_other, cpdist(nn[di], nn[o_di], scorer=fuzz.token_set_ratio,
                                                      workers=-1, dtype=np.float32) / 100.0, np.nan)
    G["g_top_addr_tset"] = np.where(has_other, cpdist(na[di], na[o_di], scorer=fuzz.token_set_ratio,
                                                      workers=-1, dtype=np.float32) / 100.0, np.nan)
    G["g_top_addr_eq"] = np.where(has_other, (na[di] == na[o_di]) & (na[di] != ""), np.nan)
    for col in ("pc", "house"):
        if col in s23:
            v = s23[col].to_numpy()
            a, b = v[di], v[o_di]
            G[f"g_top_{col}_eq"] = np.where(has_other & (a != "") & (b != ""), a == b, np.nan)
    G["g_top_p"] = np.where(has_other, p[np.maximum(other, 0)], np.nan)
    # strongest stage-1 score among OTHER candidates with the same normalised address
    addr = na[di]
    codes = pd.factorize(pd.Series(qi).astype(str) + "\x00" + pd.Series(addr))[0]
    _, n_a, _, bo_a = _side_stats(codes, p)
    G["g_same_addr_n"] = np.where(addr != "", n_a - 1, 0)
    G["g_same_addr_max"] = np.where((addr != "") & (n_a > 1), bo_a, np.nan)
    return G.astype(np.float32)


def s23_side_features(cands, p_col):
    """Competition for the same Source 2/3 record. Needs ALL candidates of that record, so in chunked
    inference compute it after every chunk has been scored."""
    r, n, m, bo = _side_stats(cands["di"].to_numpy(), cands[p_col].to_numpy(np.float64))
    return pd.DataFrame({"g_rank_s23": r, "g_n_s23": n, "g_margin_s23": m, "g_best_other_s23": bo},
                        index=cands.index).astype(np.float32)


STAGE2_BASE = ["name_tset", "name_wr", "core_tset", "core_exact", "addr_tset", "addr_core_tset", "pc_eq",
               "house_eq", "num_jacc", "name_idfcos", "addr_idfcos", "blk_text", "blk_name", "n_blockers",
               "name_soft_min", "addr_soft_max", "distinct_jacc", "name_conflict", "is_s3"]


def stage2_matrix(X1, p1, G1, G23):
    """Stage-2 design matrix: stage-1 score + key raw features + both group blocks."""
    base = X1[[c for c in STAGE2_BASE if c in X1.columns]].reset_index(drop=True)
    out = pd.concat([base, G1.reset_index(drop=True), G23.reset_index(drop=True)], axis=1)
    out.insert(0, "p1", np.asarray(p1, np.float32))
    return out

In [ ]:
"""
sampling.py - density-preserving training sample.

Taking the first N rows (or random rows) of Source 1 and random Source 2/3 negatives gives a sparse
pool: the model rarely sees the look-alike neighbours (same street, same chain, same surname) that
fill every candidate list at test time, so validation F0.5 is inflated and thresholds are too loose.

Instead take EVERY record of a few whole regions (US states / Indian states), in every source, plus
all ground-truth matches of the chosen Source 1 records (so recall is measured honestly).
Region tags are matched on the raw address in all common spellings (code, name, native script).
"""
import re

import pandas as pd

REGION_PATTERNS = {
    # US: trailing state code or name
    "OR": ("US", r",\s*(?:OR|Oregon)\s*$"),
    "KY": ("US", r",\s*(?:KY|Kentucky)\s*$"),
    "AR": ("US", r",\s*(?:AR|Arkansas)\s*$"),
    "MO": ("US", r",\s*(?:MO|Missouri)\s*$"),
    "WI": ("US", r",\s*(?:WI|Wisconsin)\s*$"),
    # India: state names, codes and native-script names anywhere in the address
    "KERALA": ("India", r"kerala|,\s*KL\b|കേരളം"),
    "PUNJAB": ("India", r"punjab|panjab|,\s*PB\b|ਪੰਜਾਬ"),
    "HARYANA": ("India", r"haryana|hariyana|,\s*HR\b|हरियाणा"),
    "ORISSA": ("India", r"orissa|odisha|,\s*OD\b|ଓଡ଼ିଶା"),
    "RAJASTHAN": ("India", r"rajasthan|,\s*RJ\b|राजस्थान"),
}
DEFAULT_REGIONS = ("OR", "KY", "AR", "KERALA", "PUNJAB", "HARYANA")


def _mask(df, regions):
    m = pd.Series(False, index=df.index)
    ctry = df["country"].astype(str)
    addr = df["business_address"].astype(str)
    for r in regions:
        c, pat = REGION_PATTERNS[r]
        m |= (ctry == c) & addr.str.contains(pat, flags=re.I, regex=True)
    return m


def region_sample(s1, s2, s3, gt, regions=DEFAULT_REGIONS):
    """Returns (s1, s23, gt) restricted to whole regions (+ all GT matches of the chosen S1)."""
    s1r = s1[_mask(s1, regions)].reset_index(drop=True)
    ids = set(s1r["entity_id"])
    gtr = gt[gt.iloc[:, 0].isin(ids)].copy()
    need = {x.strip() for m in gtr.iloc[:, 1].fillna("") for x in str(m).split(",") if x.strip()}
    parts = [d[_mask(d, regions) | d["entity_id"].isin(need)] for d in (s2, s3)]
    s23 = pd.concat(parts, ignore_index=True).drop_duplicates("entity_id").reset_index(drop=True)
    return s1r, s23, gtr

In [ ]:
"""
two_stage.py - candidate generation, two-stage GBDT matcher (LightGBM or XGBoost) and chunked, memory-safe inference.

Stage 1: pair classifier on pairwise features (features.pair_features).
Stage 2: re-scores each pair with group context (groups.py): rank/margin of the stage-1 score inside
         the Source 1 group AND among all Source 1 entities competing for the same Source 2/3 record.
Selection: two thresholds (tau1 for each Source 1's best candidate, tau2 for the others) and a GLOBAL
         one-to-one constraint (each Source 2/3 record goes to at most one Source 1).

Inference runs in Source 1 chunks. Pass 1 caches each chunk's stage-2 inputs (float16) on disk;
the Source 2/3-side group features need every chunk's stage-1 scores, so they are computed once
globally; pass 2 applies stage 2 chunk by chunk; selection is global.
"""
import gc
import os
import shutil
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

LGB_PARAMS = dict(n_estimators=1500, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, min_child_samples=40, random_state=42, n_jobs=-1, verbose=-1)


BLOCKERS = (("name", 25), ("core", 15), ("text", 20))  # (view, top-k per Source 1)
USE_GPU = os.environ.get("ER_USE_GPU", "1") != "0"


def _gpu():
    """cupy + cupyx.scipy.sparse if a CUDA device is usable, else None."""
    if not USE_GPU:
        return None
    try:
        import cupy as cp
        import cupyx.scipy.sparse as cs
        if cp.cuda.runtime.getDeviceCount() < 1:
            return None
        cp.arange(2).sum()  # compiles a kernel: fails fast if CUDA headers/driver are unusable
        return cp, cs
    except Exception as e:  # pragma: no cover
        print(f"  (GPU unavailable, blocking on CPU: {type(e).__name__}: {e})")
        return None


def _view(p, name):
    if name == "name":
        return p["norm_name"].to_numpy(object)
    if name == "core":
        return p["core"].to_numpy(object)
    return (p["norm_name"] + " " + p["norm_addr"]).to_numpy(object)


def _topk_cpu(Q, DT, k, min_sim, batch=2000):
    rs, cs_, vs = [], [], []
    for b in range(0, Q.shape[0], batch):
        r, c, v = _topk_rows((Q[b:b + batch] @ DT).tocsr(), k, min_sim)
        rs.append(r.astype(np.int64) + b), cs_.append(c), vs.append(v)
    if not rs:
        return np.array([], np.int64), np.array([], np.int64), np.array([], np.float32)
    return np.concatenate(rs), np.concatenate(cs_), np.concatenate(vs).astype(np.float32)


def _topk_gpu(Q, DT, k, min_sim, gpu, batch=1024):
    """Same result as _topk_cpu (up to ties): sparse Q @ DT on the GPU, then per-row top-k by sorting
    (row asc, score desc). Halves the batch on GPU out-of-memory."""
    cp, cs = gpu
    DTg = cs.csr_matrix(DT)
    rs, cs_, vs = [], [], []
    b = 0
    try:
        while b < Q.shape[0]:
            try:
                Qb = cs.csr_matrix(Q[b:b + batch])
                S = (Qb @ DTg).tocoo()
                keep = S.data >= min_sim
                r, c, v = S.row[keep], S.col[keep], S.data[keep]
                del S, Qb
                order = cp.lexsort(cp.stack([-v, r.astype(cp.float32)]))
                r, c, v = r[order], c[order], v[order]
                rank = cp.arange(r.size) - cp.searchsorted(r, r, side="left")
                sel = rank < k
                rs.append(cp.asnumpy(r[sel]).astype(np.int64) + b)
                cs_.append(cp.asnumpy(c[sel]).astype(np.int64))
                vs.append(cp.asnumpy(v[sel]).astype(np.float32))
                del r, c, v, order, rank, sel
                b += batch
            except cp.cuda.memory.OutOfMemoryError:
                cp.get_default_memory_pool().free_all_blocks()
                if batch <= 16:
                    raise
                batch //= 2
    finally:
        del DTg
        cp.get_default_memory_pool().free_all_blocks()
    if not rs:
        return np.array([], np.int64), np.array([], np.int64), np.array([], np.float32)
    return np.concatenate(rs), np.concatenate(cs_), np.concatenate(vs)


def _gpu_devices(gpu):
    """Devices to use: all visible GPUs (override with ER_GPU_DEVICES="0,1")."""
    env = os.environ.get("ER_GPU_DEVICES")
    if env:
        return [int(x) for x in env.split(",") if x.strip()]
    return list(range(gpu[0].cuda.runtime.getDeviceCount()))


def _topk_multi_gpu(Q, DT, k, min_sim, gpu):
    """Split the Source 1 rows across GPUs (one thread per device; each gets its own copy of DT).
    CuPy work on one device does not block Python threads driving another, so devices run in parallel.
    Rows keep their order, so results are identical to a single-GPU run."""
    import threading
    cp = gpu[0]
    devs = _gpu_devices(gpu)
    if len(devs) < 2 or Q.shape[0] < 2 * 1024:
        with cp.cuda.Device(devs[0] if devs else 0):
            return _topk_gpu(Q, DT, k, min_sim, gpu)
    bounds = np.linspace(0, Q.shape[0], len(devs) + 1).astype(int)
    out, errs = [None] * len(devs), []

    def run(j):
        try:
            with cp.cuda.Device(devs[j]):
                r, c, v = _topk_gpu(Q[bounds[j]:bounds[j + 1]], DT, k, min_sim, gpu)
            out[j] = (r + bounds[j], c, v)
        except BaseException as e:  # re-raised below -> CPU fallback for this view
            errs.append(e)

    th = [threading.Thread(target=run, args=(j,)) for j in range(len(devs))]
    for t in th:
        t.start()
    for t in th:
        t.join()
    if errs:
        raise errs[0]
    return tuple(np.concatenate([o[i] for o in out]) for i in range(3))


def iter_candidate_chunks(qp, dp, chunk_size=250_000, min_sim=0.015, max_df=0.01, log=None):
    """Country-partitioned char 3-4gram TF-IDF blocking, 3 views, top-k each. Per country and view the
    vectoriser is fitted ONCE on the Source 2/3 side and all Source 1 records of that country are scored
    (on the GPU when available). Yields (qidx, cands) per Source 1 chunk, with cands.qi LOCAL to
    qp.iloc[qidx] and cands.di global."""
    gpu = _gpu()
    qc, dc = qp["country"].to_numpy(object), dp["country"].to_numpy(object)
    for g in pd.unique(qc):
        q_all = np.flatnonzero(qc == g)
        d_all = np.arange(len(dp)) if g == "" else np.flatnonzero((dc == g) | (dc == ""))
        if len(q_all) == 0 or len(d_all) == 0:
            continue
        t0 = time.time()
        views = {}
        for name, k in BLOCKERS:
            vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), min_df=2, max_df=max_df,
                                  sublinear_tf=True, dtype=np.float32)
            DT = vec.fit_transform(_view(dp.iloc[d_all], name)).T.tocsr()
            Q = vec.transform(_view(qp.iloc[q_all], name)).tocsr()
            kk = min(k, len(d_all))
            r, c, v = None, None, None
            if gpu is not None:
                try:
                    r, c, v = _topk_multi_gpu(Q, DT, kk, min_sim, gpu)
                except Exception as e:  # any GPU failure -> CPU for this view
                    print(f"  (GPU blocking failed for {g}/{name}: {type(e).__name__}; using CPU)")
            if r is None:
                r, c, v = _topk_cpu(Q, DT, kk, min_sim)
            views[name] = (r, d_all[c], v)
            del vec, DT, Q
            gc.collect()
        if log:
            log(f"  blocking {g}: {len(q_all):,} S1 x {len(d_all):,} S23 in {time.time() - t0:.0f}s "
                f"({f'{len(_gpu_devices(gpu))} GPU' if gpu else 'CPU'})")
        for s in range(0, len(q_all), chunk_size):
            e = min(s + chunk_size, len(q_all))
            blocks = {}
            for name, (r, d, v) in views.items():
                lo, hi = np.searchsorted(r, s), np.searchsorted(r, e)  # r is sorted (row-major)
                blocks[name] = pd.DataFrame({"qi": r[lo:hi] - s, "di": d[lo:hi], "sim": v[lo:hi]})
            yield q_all[s:e], union_candidates(blocks)
        del views
        gc.collect()


def block_candidates(qp, dp, log=None):
    """All candidates at once (training): cands.qi indexes qp, cands.di indexes dp."""
    out = []
    for qidx, c in iter_candidate_chunks(qp, dp, chunk_size=len(qp) or 1, log=log):
        c["qi"] = qidx[c["qi"].to_numpy()]
        out.append(c)
    if not out:
        return pd.DataFrame(columns=["qi", "di", "n_blockers"])
    c = pd.concat(out, ignore_index=True)
    blk = [x for x in c.columns if x.startswith("blk_")]
    c[blk] = c[blk].astype(np.float32)
    return c


# ------------------------------------------------------------------ model backends
BACKEND = os.environ.get("ER_BACKEND", "auto")  # "auto" (xgb on GPU, else lgbm), "lgbm" or "xgb"
XGB_PARAMS = dict(n_estimators=1500, learning_rate=0.05, max_depth=8, min_child_weight=5, subsample=0.8,
                  colsample_bytree=0.8, tree_method="hist", max_bin=256, eval_metric="logloss",
                  random_state=42, n_jobs=-1)


def _xgb_device():
    try:
        import cupy as cp
        return "cuda" if USE_GPU and cp.cuda.runtime.getDeviceCount() > 0 else "cpu"
    except Exception:
        return "cpu"


class _Model:
    """Uniform wrapper: fit / predict_proba / best_iteration / feature_importance / cols."""

    def __init__(self, backend, n_estimators=None, **over):
        self.backend = backend
        if backend == "xgb":
            import xgboost as xgb
            p = {**XGB_PARAMS, **{k: v for k, v in over.items() if k in XGB_PARAMS}}
            if n_estimators:
                p["n_estimators"] = n_estimators
            self.m = xgb.XGBClassifier(device=_xgb_device(), **p)
        else:
            p = {**LGB_PARAMS, **over}
            if n_estimators:
                p["n_estimators"] = n_estimators
            self.m = lgb.LGBMClassifier(**p)

    def fit(self, X, y, Xv=None, yv=None):
        self.cols = list(X.columns)
        if self.backend == "xgb":
            if Xv is not None:
                self.m.set_params(early_stopping_rounds=50)
                self.m.fit(X, y, eval_set=[(Xv, yv)], verbose=False)
            else:
                self.m.fit(X, y, verbose=False)
            self.best_iteration_ = (self.m.best_iteration + 1) if Xv is not None else self.m.n_estimators
        else:
            if Xv is not None:
                self.m.fit(X, y, eval_set=[(Xv, yv)], callbacks=[lgb.early_stopping(50, verbose=False)])
            else:
                self.m.fit(X, y)
            self.best_iteration_ = self.m.best_iteration_ or self.m.n_estimators
        return self

    def predict_proba(self, X):
        return self.m.predict_proba(X[self.cols])

    def importance(self):
        if self.backend == "xgb":
            g = self.m.get_booster().get_score(importance_type="total_gain")
            return pd.Series({c: g.get(c, 0.0) for c in self.cols}).sort_values(ascending=False)
        return pd.Series(self.m.booster_.feature_importance("gain"), index=self.cols).sort_values(ascending=False)


def resolve_backend(backend=None):
    """A/B on whole training regions: XGBoost 0.9786 vs LightGBM 0.9783 held-out F0.5 (equal within noise);
    XGBoost is much faster on a GPU, LightGBM on CPU-only machines."""
    b = backend or BACKEND
    if b == "auto":
        try:
            import xgboost  # noqa: F401
            b = "xgb" if _xgb_device() == "cuda" else "lgbm"
        except Exception:
            b = "lgbm"
    return b


def _fit(X, y, Xv=None, yv=None, n_estimators=None, backend=None, **over):
    return _Model(resolve_backend(backend), n_estimators=n_estimators, **over).fit(X, y, Xv, yv)


def train_two_stage(cands, X, y, n_true, s23p, seed=42, folds=5, log=print):
    """cands: qi, di (+ blocker cols). X: stage-1 features. y: labels. n_true: true match count per S1.
    Split by Source 1: 60% fit, 20% early stopping + threshold tuning, 20% held-out report."""
    qi = cands["qi"].to_numpy()
    n_s1 = len(n_true)
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_s1)
    part = np.empty(n_s1, np.int8)
    part[perm[: int(0.6 * n_s1)]] = 0
    part[perm[int(0.6 * n_s1): int(0.8 * n_s1)]] = 1
    part[perm[int(0.8 * n_s1):]] = 2
    pq = part[qi]
    tr, va = pq == 0, pq == 1

    t0 = time.time()
    clf1 = _fit(X[tr], y[tr], X[va], y[va])
    p1 = clf1.predict_proba(X)[:, 1].astype(np.float32)
    n_iter = clf1.best_iteration_
    fold = (qi.astype(np.int64) * 2654435761) % folds
    for k in range(folds):  # out-of-fold stage-1 scores on the fit part (stage 2 must not see leaked scores)
        a, b = tr & (fold != k), tr & (fold == k)
        p1[b] = _fit(X[a], y[a], n_estimators=n_iter).predict_proba(X[b])[:, 1]
    log(f"  stage 1 [{resolve_backend()}]: {n_iter} trees, OOF done ({time.time() - t0:.0f}s)")

    c = cands[["qi", "di"]].copy()
    c["p1"] = p1
    X2 = stage2_matrix(X, p1, s1_side_features(c, "p1", s23p), s23_side_features(c, "p1"))
    clf2 = _fit(X2[tr], y[tr], X2[va], y[va], num_leaves=31)
    score = clf2.predict_proba(X2)[:, 1]
    log(f"  stage 2: {clf2.best_iteration_} trees ({time.time() - t0:.0f}s)")

    c["y"], c["score"] = y, score
    val_ids, test_ids = np.flatnonzero(part == 1), np.flatnonzero(part == 2)
    f_val, t1, t2 = tune_policy(c[va], "score", n_true, val_ids, one_to_one=True)
    sel = select_matches(c[pq == 2], "score", t1, t2, one_to_one=True)
    f_test = macro_f05(sel, n_true, test_ids)
    # stage 1 alone, for reference
    c["s1score"] = clf1.predict_proba(X)[:, 1]
    f1v, a1, a2 = tune_policy(c[va], "s1score", n_true, val_ids, one_to_one=True)
    f1t = macro_f05(select_matches(c[pq == 2], "s1score", a1, a2, one_to_one=True), n_true, test_ids)
    log(f"  held-out Macro F0.5: stage 1 = {f1t:.4f}, two-stage = {f_test:.4f} (tau1={t1:.2f}, tau2={t2:.2f})")
    imp = clf2.importance()
    return {"clf1": clf1, "clf2": clf2, "t1": t1, "t2": t2, "f05_val": f_val, "f05_test": f_test,
            "f05_test_stage1": f1t, "stage2_cols": list(X2.columns), "stage2_importance": imp,
            "part": part, "pairs": c}


def predict_chunked(test_s1p, test_s23p, model, chunk_size=250_000, cache_dir="stage2_cache", log=print,
                    workers=-1):
    """Returns (cands, selected): DataFrames with global qi (into test_s1p) and di (into test_s23p)."""
    os.makedirs(cache_dir, exist_ok=True)
    clf1, clf2 = model["clf1"], model["clf2"]
    cols = model["stage2_cols"]
    g23_cols = [c for c in cols if c.endswith("_s23") and c.startswith("g_")]
    chunks = []
    t0 = time.time()
    for ci, (qidx, cc) in enumerate(iter_candidate_chunks(test_s1p, test_s23p, chunk_size=chunk_size, log=log)):
        if len(cc) == 0:
            continue
        cs = test_s1p.iloc[qidx].reset_index(drop=True)
        X = pair_features(cc, cs, test_s23p, workers=workers)
        p1 = clf1.predict_proba(X)[:, 1].astype(np.float32)
        cc = cc[["qi", "di"]].copy()
        cc["p1"] = p1
        G1 = s1_side_features(cc, "p1", test_s23p)
        part = stage2_matrix(X, p1, G1, pd.DataFrame(index=cc.index))
        part = part.reindex(columns=[c for c in cols if c not in g23_cols])
        path = os.path.join(cache_dir, f"chunk{ci}.npy")
        np.save(path, part.to_numpy(np.float16))
        chunks.append((path, list(part.columns), qidx[cc["qi"].to_numpy()].astype(np.int32),
                       cc["di"].to_numpy().astype(np.int32), p1))
        log(f"  [pass 1] chunk {ci + 1}: {len(qidx):,} S1 ({cs['country'].iat[0]}), {len(cc):,} candidates "
            f"({time.time() - t0:.0f}s)")
        del cs, cc, X, G1, part
        gc.collect()

    if not chunks:
        empty = pd.DataFrame({"qi": np.array([], np.int32), "di": np.array([], np.int32)})
        return empty, empty
    qi = np.concatenate([c[2] for c in chunks])
    di = np.concatenate([c[3] for c in chunks])
    p1 = np.concatenate([c[4] for c in chunks])
    G23 = s23_side_features(pd.DataFrame({"di": di, "p1": p1}), "p1")
    score = np.empty(len(qi), np.float32)
    off = 0
    for path, pcols, cqi, _, _ in chunks:
        n = len(cqi)
        M = pd.DataFrame(np.load(path).astype(np.float32), columns=pcols)
        for c in g23_cols:
            M[c] = G23[c].to_numpy()[off:off + n]
        score[off:off + n] = clf2.predict_proba(M[cols])[:, 1]
        off += n
        os.remove(path)
    shutil.rmtree(cache_dir, ignore_errors=True)
    log(f"  [pass 2] stage 2 scored {len(qi):,} pairs ({time.time() - t0:.0f}s)")
    cands = pd.DataFrame({"qi": qi, "di": di, "score": score})
    selected = select_matches(cands, "score", model["t1"], model["t2"], one_to_one=True)
    return cands, selected


def id_lists(n_s1, qi, di, s23_ids):
    """One comma-joined, de-duplicated, sorted ID string per Source 1 row (empty when none)."""
    out = np.full(n_s1, "", dtype=object)
    if len(qi) == 0:
        return out
    ids = np.asarray(s23_ids, dtype=object)[di]
    order = np.argsort(qi, kind="stable")
    q_sorted, ids_sorted = qi[order], ids[order]
    starts = np.flatnonzero(np.r_[True, q_sorted[1:] != q_sorted[:-1]])
    ends = np.r_[starts[1:], len(q_sorted)]
    for s, e in zip(starts, ends):
        out[q_sorted[s]] = ",".join(sorted(set(ids_sorted[s:e])))
    return out

In [ ]:
# [EXECUTION 1] Density-preserving training sample -> normalisation -> blocking -> two-stage GBDT (XGBoost on GPU)
T0 = time.time()
TRAIN_REGIONS = ("PUNJAB",) if DEV_MODE else DEFAULT_REGIONS
print("Loading training data...")
s1 = read_tsv(train_s1_path); s2 = read_tsv(train_s2_path); s3 = read_tsv(train_s3_path); gt = read_tsv(train_gt_path)
s1, s23, gt = region_sample(s1, s2, s3, gt, TRAIN_REGIONS)
del s2, s3; gc.collect()
gt_dict = parse_gt(gt)
print(f"Training regions {TRAIN_REGIONS}: S1={len(s1):,}, S23={len(s23):,}, true pairs={sum(map(len, gt_dict.values())):,}")
diag = gt_diagnostics(gt_dict, s1, s23)
for k in ("singleton_share", "singleton_share_by_country", "s23_claimed_by_more_than_one_s1"):
    print(f"  {k}: {diag[k]}")

s1p = prepare_side(s1); s23p = prepare_side(s23)
print(f"Prepared training tables ({time.time()-T0:.0f}s)")
cands = block_candidates(s1p, s23p, log=print)
s1_ids, s23_ids = s1p["entity_id"].to_numpy(), s23p["entity_id"].to_numpy()
labels = np.array([s23_ids[d] in gt_dict.get(s1_ids[q], ()) for q, d in zip(cands.qi, cands.di)], np.int32)
n_true = np.array([len(gt_dict.get(e, ())) for e in s1_ids])
recall = labels.sum() / max(n_true.sum(), 1)
print(f"Candidates: {len(cands):,}  blocking recall: {recall*100:.2f}%  ({time.time()-T0:.0f}s)")

X_train = pair_features(cands, s1p, s23p, workers=-1)
print(f"Feature matrix: {X_train.shape}  ({time.time()-T0:.0f}s)")
model = train_two_stage(cands, X_train, labels, n_true, s23p)
best_f05, t1, t2 = model["f05_test"], model["t1"], model["t2"]
print("Top stage-2 features:", ", ".join(model["stage2_importance"].head(10).index))
del X_train, cands, s1p, s23p, s1, s23, gt; gc.collect()
print(f"Training done in {time.time()-T0:.0f}s")

In [ ]:
# [EXECUTION 2] Test inference: pass 1 (blocking, features, stage 1) per chunk -> global group features
# -> pass 2 (stage 2) -> globally one-to-one selection
T1 = time.time()
test_s1 = read_tsv(test_s1_path); test_s2 = read_tsv(test_s2_path); test_s3 = read_tsv(test_s3_path)
if DEV_MODE:
    test_s1, test_s2, test_s3 = test_s1.head(2000), test_s2.head(20000), test_s3.head(20000)
test_s23 = pd.concat([test_s2, test_s3], ignore_index=True)
del test_s2, test_s3; gc.collect()
print(f"Test: S1={len(test_s1):,}, S23={len(test_s23):,}")
print("Countries:", test_s1["country"].value_counts().to_dict())

SLIM = ("business_name", "business_address", "ml_addr")   # raw text not needed after cleaning
test_s1p = prepare_side(test_s1, drop=SLIM)
test_s23p = prepare_side(test_s23, drop=SLIM)
del test_s23; gc.collect()
print(f"Prepared test tables ({time.time()-T1:.0f}s)")

CHUNK_SIZE = 1_000 if DEV_MODE else 250_000
test_cands, test_sel = predict_chunked(test_s1p, test_s23p, model, chunk_size=CHUNK_SIZE,
                                       cache_dir=str(WORKING_DIR / "stage2_cache"))

ts1_ids, ts23_ids = test_s1p["entity_id"].to_numpy(), test_s23p["entity_id"].to_numpy()
cand_out = pd.DataFrame({"source1_entity_id": ts1_ids,
                         "candidate_entity_ids": id_lists(len(ts1_ids), test_cands.qi.to_numpy(), test_cands.di.to_numpy(), ts23_ids)})
match_out = pd.DataFrame({"source1_entity_id": ts1_ids,
                          "matched_entity_ids": id_lists(len(ts1_ids), test_sel.qi.to_numpy(), test_sel.di.to_numpy(), ts23_ids)})
cand_file = OUTPUT_DIR / "candidate_pairs.tsv"
match_file = OUTPUT_DIR / "matching_results.tsv"
cand_out.to_csv(cand_file, sep="\t", index=False)
match_out.to_csv(match_file, sep="\t", index=False)

assert len(match_out) == len(test_s1) and match_out["source1_entity_id"].is_unique
assert len(cand_out) == len(test_s1)
n_matched = (match_out["matched_entity_ids"] != "").sum()
print(f"Saved {match_file.name}: {len(match_out):,} rows, {n_matched:,} non-empty")
print(f"Saved {cand_file.name}: {len(cand_out):,} rows, {len(test_cands):,} candidate pairs")
print("Non-empty share by country:", (match_out["matched_entity_ids"] != "").groupby(test_s1p["country"].to_numpy()).mean().round(3).to_dict())
print(f"Inference done in {time.time()-T1:.0f}s")

In [ ]:
# [PACKAGING] Metrics, methodology document & submission zips
metrics_data = {
    "heldout_macro_f05_two_stage": round(float(model["f05_test"]), 4),
    "heldout_macro_f05_stage1_only": round(float(model["f05_test_stage1"]), 4),
    "tau1": round(float(t1), 2), "tau2": round(float(t2), 2),
    "train_blocking_recall": round(float(recall), 4),
    "train_regions": list(TRAIN_REGIONS),
    "model_backend": resolve_backend(), "gpu_blocking": _gpu() is not None,
    "test_s1_count": len(test_s1), "test_candidate_pairs": int(len(test_cands)), "test_matches": int(len(test_sel)),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}
with open(WORKING_DIR / "metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=2)
print(json.dumps(metrics_data, indent=2))

_backend_name = {"xgb": "XGBoost (GPU)", "lgbm": "LightGBM"}.get(resolve_backend(), resolve_backend())
methodology_content = f"""# Business Entity Resolution - Methodology
### Team: zamzon_ai

## Summary
Multilingual normalisation with an offline-built static lexicon, 3-view country-partitioned TF-IDF
blocking, a two-stage {_backend_name} matcher (pair features, then group context) and a two-threshold,
globally one-to-one selection. Held-out Macro F0.5 on whole training regions: **{model['f05_test']:.4f}**
(stage 1 alone: {model['f05_test_stage1']:.4f}); training blocking recall {recall*100:.2f}%.

## 1. Normalisation
- Every Indic script + Urdu romanised (no names wiped), French ligatures/apostrophes handled, legal forms,
  street types, landmarks (near/opp/ke paas/en face de), PIN/ZIP parsing.
- Static lexicon (src/resources/*.tsv), applied per country (unknown countries use the script-level part):
  English words written in Indic scripts (kansaltents->consultants), abbreviations (r->rue, mh->maharashtra,
  state codes), state/phrase variants (tamil nadu->tn), frequent typos, OCR digit noise (hea1th->health),
  junk address tokens (null, na, pmb).
- The lexicon was built once, offline, from word statistics: candidates from training ground-truth swaps and
  vocabulary counts, judged word-by-word by an evaluation model (TypeSafe Jev). It only saw single words or
  short phrases, never a record, and never decided whether two businesses match. It is not called at run
  time; the pipeline reads static TSV files. Every rule was checked against training labels where they exist.

## 2. Blocking
Country-partitioned char 3-4gram TF-IDF (max_df=0.01, sparse products on the GPU via CuPy when available) on name (top 25), core name (top 15) and
name+address (top 20); union. Vectorisers are fitted once per country on Source 2/3.
Test: {len(test_cands):,} candidate pairs for {len(test_s1):,} Source 1 entities.

## 3. Matching model
- Stage 1: {_backend_name} on ~57 pair features (fuzzy name/core/address ratios, IDF cosines, postcode / house /
  unit agreement, phonetic keys, DBA and landmark handling, distinctive-word similarity, look-alike name
  conflict flag, blocker similarities).
- Stage 2: {_backend_name} on the stage-1 score plus group context: rank and margin inside the Source 1 group and
  among all Source 1 entities competing for the same Source 2/3 record, similarity to the best other
  candidate, same-address support. Stage-1 scores for stage-2 training are out-of-fold.
- Training data: every record of whole regions ({', '.join(TRAIN_REGIONS)}) to keep test-like density.
  Split by Source 1: 60% fit / 20% early stopping + threshold tuning / 20% held-out report.

## 4. Decision policy
tau1 = {t1:.2f} (best candidate), tau2 = {t2:.2f} (additional candidates); each Source 2/3 record is assigned
to at most one Source 1 entity (global one-to-one, matching the ground-truth structure).
"""
doc_path = WORKING_DIR / "methodology_document.md"
doc_path.write_text(methodology_content, encoding="utf-8")

sub_zip = WORKING_DIR / "submission_outputs.zip"
with zipfile.ZipFile(sub_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(match_file, arcname="output/matching_results.tsv")
    zf.write(cand_file, arcname="output/candidate_pairs.tsv")
    zf.write(doc_path, arcname="methodology_document.md")
print(f"Ready: {sub_zip.name} ({sub_zip.stat().st_size:,} bytes)")

In [ ]:
# [VALIDATION] Official Submission Validation
if val_script and Path(val_script).exists():
    res = subprocess.run([sys.executable, str(val_script), "--matching", str(match_file),
                          "--candidate", str(cand_file), "--test-dir", str(test_s1_path.parent)],
                         capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print("VALIDATION ERRORS:")
        print(res.stderr)
else:
    print(f"Validator not found; row-count assertions passed ({len(match_out):,} rows).")